In [1]:
# %% Cell 1: Imports and Setup
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import os
import json
from datetime import datetime

# Seed everything for reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)


In [2]:

# %% Cell 2: Data Loading and Preprocessing (Run Once)
def load_and_preprocess_data(file_path, target_pollutant):
    """
    Load dataset and prepare for time series forecasting
    Returns scaled data with timestep structuring
    """
    # Load data
    df = pd.read_csv(file_path, parse_dates=['Timestamp'], index_col='Timestamp')
    
    # Select target pollutant and relevant features
    pollutants = [
        'PM2.5 (µg/m³)', 'PM10 (µg/m³)', 'NO (µg/m³)', 
        'NO2 (µg/m³)', 'SO2 (µg/m³)', 'CO (mg/m³)', 
        'Ozone (µg/m³)'
    ]
    df = df[pollutants].resample('15T').mean().ffill()
    
    # Create sliding window dataset
    def create_dataset(data, n_steps=1):
        X, y = [], []
        for i in range(len(data)-n_steps):
            X.append(data[i:(i+n_steps), :])
            y.append(data[i + n_steps, pollutants.index(target_pollutant)])
        return np.array(X), np.array(y)
    
    # Scale data
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(df)
    
    # Create time steps
    n_steps = 4  # 1 hour window (4*15min)
    X, y = create_dataset(scaled_data, n_steps)
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, shuffle=False, random_state=SEED
    )
    
    return (X_train, y_train), (X_test, y_test), scaler, df.shape[1]


In [3]:

# %% Cell 3: Model Architectures
def build_hybrid_model(input_shape, mlp_layers, lstm_layers, lstm_units, bidirectional=False):
    """
    Build hybrid MLP-LSTM model with configurable architecture
    """
    model = tf.keras.Sequential()
    
    # MLP Branch
    model.add(tf.keras.layers.Flatten(input_shape=input_shape))
    for units in mlp_layers:
        model.add(tf.keras.layers.Dense(units, activation='relu'))
        model.add(tf.keras.layers.Dropout(0.2))
    
    # Reshape for LSTM
    model.add(tf.keras.layers.Reshape((1, -1)))
    
    # LSTM Branch
    for i in range(lstm_layers):
        return_sequences = i < (lstm_layers - 1)
        if bidirectional:
            model.add(tf.keras.layers.Bidirectional(
                tf.keras.layers.LSTM(lstm_units, return_sequences=return_sequences)
            ))
        else:
            model.add(tf.keras.layers.LSTM(lstm_units, return_sequences=return_sequences))
        model.add(tf.keras.layers.Dropout(0.2))
    
    # Final Output
    model.add(tf.keras.layers.Dense(1))
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    return model


In [40]:

# %% Cell 4: Hyperparameter Configurations
POLLUTANTS = [
    'PM2.5 (µg/m³)', 'PM10 (µg/m³)', 'NO (µg/m³)',
    'NO2 (µg/m³)', 'SO2 (µg/m³)', 'CO (mg/m³)', 
    'Ozone (µg/m³)'
]

CONFIGURATIONS = [
    # MLP Layers, LSTM Layers, LSTM Units, Bidirectional
    ([64, 32], 1, 128, True),
    ([128], 2, 64, False),
    ([256, 128], 1, 256, True),
    ([64, 64], 2, 128, False),
    ([128, 64, 32], 1, 64, True),
]


In [41]:

# %% Cell 5: Training Loop with Checkpointing
def run_experiment(data_path, results_file='model_results.csv'):
    # Create results file if not exists
    if not os.path.exists(results_file):
        pd.DataFrame(columns=[
            'timestamp', 'pollutant', 'mlp_layers', 'lstm_layers',
            'lstm_units', 'bidirectional', 'mse', 'mae'
        ]).to_csv(results_file, index=False)
    
    for pollutant in POLLUTANTS:
        # Load data for current pollutant
        (X_train, y_train), (X_test, y_test), scaler, n_features = load_and_preprocess_data(
            data_path, pollutant
        )
        
        for config in CONFIGURATIONS:
            mlp_layers, lstm_layers, lstm_units, bidirectional = config
            
            # Skip already tested configurations
            existing = pd.read_csv(results_file)
            mask = (existing['pollutant'] == pollutant) & \
                   (existing['mlp_layers'].astype(str) == str(mlp_layers)) & \
                   (existing['lstm_layers'] == lstm_layers) & \
                   (existing['lstm_units'] == lstm_units) & \
                   (existing['bidirectional'] == bidirectional)
            if not mask.any():
                # Build and train model
                model = build_hybrid_model(
                    input_shape=(X_train.shape[1], X_train.shape[2]),
                    mlp_layers=mlp_layers,
                    lstm_layers=lstm_layers,
                    lstm_units=lstm_units,
                    bidirectional=bidirectional
                )
                
                early_stop = tf.keras.callbacks.EarlyStopping(
                    monitor='val_loss', patience=5, restore_best_weights=True
                )
                
                history = model.fit(
                    X_train, y_train,
                    validation_split=0.2,
                    epochs=100,
                    batch_size=32,
                    callbacks=[early_stop],
                    verbose=0
                )
                
                # Evaluate
                mse, mae = model.evaluate(X_test, y_test, verbose=0)
                
                # Save results
                new_row = pd.DataFrame([{
                    'timestamp': datetime.now().isoformat(),
                    'pollutant': pollutant,
                    'mlp_layers': str(mlp_layers),
                    'lstm_layers': lstm_layers,
                    'lstm_units': lstm_units,
                    'bidirectional': bidirectional,
                    'mse': mse,
                    'mae': mae
                }])
                
                new_row.to_csv(results_file, mode='a', header=False, index=False)
                print(f"Saved results for {pollutant} - {config}")


In [42]:

# %% Cell 6: Run the Experiment (Execute This Cell)
DATA_PATH = "Non_Null_Datasets/preprocessed_Raw_data_15Min_2024_site_5024_Alipur_Delhi_DPCC_15Min.csv"
run_experiment(DATA_PATH)


C:\Users\DELL\AppData\Local\Temp\ipykernel_420\2707660949.py:16: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df = df[pollutants].resample('15T').mean().ffill()
C:\Users\DELL\AppData\Local\Temp\ipykernel_420\2707660949.py:16: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df = df[pollutants].resample('15T').mean().ffill()
C:\Users\DELL\AppData\Local\Temp\ipykernel_420\2707660949.py:16: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df = df[pollutants].resample('15T').mean().ffill()
C:\Users\DELL\AppData\Local\Temp\ipykernel_420\2707660949.py:16: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df = df[pollutants].resample('15T').mean().ffill()
C:\Users\DELL\AppData\Local\Temp\ipykernel_420\2707660949.py:16: FutureWarning: 'T' is deprecated and will be removed in a futur

In [43]:

# %% Cell 7: Generate Report (Run After Experiment)
def generate_report(results_file='model_results.csv', report_file='model_report.csv'):
    results = pd.read_csv(results_file)
    report = []
    
    for pollutant in POLLUTANTS:
        subset = results[results['pollutant'] == pollutant]
        best_model = subset.loc[subset['mse'].idxmin()]
        
        report.append({
            'Pollutant': pollutant,
            'Best_Configuration': f"MLP{best_model['mlp_layers']} + {'Bi' if best_model['bidirectional'] else ''}LSTM({best_model['lstm_layers']}x{best_model['lstm_units']})",
            'MSE': best_model['mse'],
            'MAE': best_model['mae'],
            'Training_Date': best_model['timestamp']
        })
    
    report_df = pd.DataFrame(report)
    report_df.to_csv(report_file, index=False)
    return report_df

# Display report
generate_report()

,Pollutant,Best_Configuration,MSE,MAE,Training_Date
0,PM2.5 (µg/m³),MLP[128] + LSTM(2x64),0.000454,0.011243,2025-02-06T15:20:17.477448
1,PM10 (µg/m³),MLP[128] + LSTM(2x64),0.000563,0.011233,2025-02-06T15:22:53.657585
2,NO (µg/m³),"MLP[256, 128] + BiLSTM(1x256)",0.000074,0.003616,2025-02-06T15:27:24.358690
3,NO2 (µg/m³),"MLP[128, 64, 32] + BiLSTM(1x64)",0.000253,0.010107,2025-02-06T15:31:20.647216
4,SO2 (µg/m³),MLP[128] + LSTM(2x64),0.000060,0.005580,2025-02-06T15:32:25.773451
5,CO (mg/m³),MLP[128] + LSTM(2x64),0.000219,0.010473,2025-02-06T15:36:34.038071
6,Ozone (µg/m³),MLP[128] + LSTM(2x64),0.000910,0.014499,2025-02-06T15:39:53.249990


In [44]:
import numpy as np
import pandas as pd
import tensorflow as tf
import os
import ast
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# -----------------------------
# RE-USE THE FUNCTIONS YOU ALREADY DEFINED:
#   1) load_and_preprocess_data()
#   2) build_hybrid_model()
#   3) POLLUTANTS list
#   4) The hyperparameter structure, if needed
# Make sure you have them in the same notebook or imported from a module.
# -----------------------------

def evaluate_on_new_data(
    results_file='model_results.csv',
    new_data_folder='Non_null_datasets',
    output_file='new_data_evaluation.csv'
):
    """
    1. Finds best (lowest MSE) hyperparameters for each pollutant from results_file.
    2. For each pollutant, retrains a new model using the best hyperparameters on the original dataset.
    3. Loads each new CSV in `new_data_folder` (except the one used for training) and evaluates.
    4. Saves a final report with MSE, MAE for each pollutant-csv combination.
    """
    
    # Step A: Read all experiment results
    all_results = pd.read_csv(results_file)
    
    # Prepare a DataFrame to hold evaluation results on new data
    columns = ['timestamp', 'pollutant', 'new_csv', 'mse', 'mae']
    evaluation_records = []
    
    # Identify the 4 new CSVs (and exclude the training CSV if you know its exact filename)
    all_csvs = [f for f in os.listdir(new_data_folder) 
                if f.startswith("preprocessed") and f.endswith(".csv")]
    
    # If you want to exclude the original training file, do so here:
    # training_file = "preprocessed_Raw_data_15Min_2024_site_5024_Alipur_Delhi_DPCC_15Min.csv"
    # new_csvs = [f for f in all_csvs if f != training_file]
    
    # Or if you have 4 known CSVs by a pattern, you can just keep them all:
    new_csvs = all_csvs  # Adjust this as needed
    
    # Step B: Loop through each pollutant to get best config
    for pollutant in POLLUTANTS:
        # Subset results for this pollutant
        subset = all_results[all_results['pollutant'] == pollutant]
        
        # Find row with the best (lowest) MSE
        best_row = subset.loc[subset['mse'].idxmin()]
        
        # Parse out best hyperparameters
        mlp_layers = ast.literal_eval(best_row['mlp_layers'])  # convert string "[64, 32]" -> Python list
        lstm_layers = int(best_row['lstm_layers'])
        lstm_units = int(best_row['lstm_units'])
        bidirectional = bool(best_row['bidirectional'])
        
        # Step C: Reload and preprocess the *original* training data for this pollutant
        # (Same file you used in your run_experiment)
        original_csv = "Non_Null_Datasets/preprocessed_Raw_data_15Min_2024_site_5024_Alipur_Delhi_DPCC_15Min.csv"
        (X_train, y_train), (X_test, y_test), scaler, n_features = load_and_preprocess_data(
            original_csv, pollutant
        )
        
        # Combine train + test for a final training step, if you prefer:
        #   Or just train on X_train if you want to match your experiment’s approach exactly.
        # Here, let's do the standard approach: train only on X_train
        # (If you want to finalize on all data, you can do: 
        #  X_train_final = np.concatenate([X_train, X_test], axis=0)
        #  y_train_final = np.concatenate([y_train, y_test], axis=0)
        #  then train on those.)
        
        # Build best model architecture
        model = build_hybrid_model(
            input_shape=(X_train.shape[1], X_train.shape[2]),
            mlp_layers=mlp_layers,
            lstm_layers=lstm_layers,
            lstm_units=lstm_units,
            bidirectional=bidirectional
        )
        
        # Train the model
        early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        model.fit(
            X_train, y_train,
            validation_split=0.2,
            epochs=100,
            batch_size=32,
            callbacks=[early_stop],
            verbose=0
        )
        
        # Step D: Evaluate on each new CSV
        for csv_file in new_csvs:
            new_csv_path = os.path.join(new_data_folder, csv_file)
            
            # Load the new data
            df_new = pd.read_csv(new_csv_path, parse_dates=['Timestamp'], index_col='Timestamp')
            
            # The same columns you used in training
            pollutants_cols = [
                'PM2.5 (µg/m³)', 'PM10 (µg/m³)', 'NO (µg/m³)', 
                'NO2 (µg/m³)', 'SO2 (µg/m³)', 'CO (mg/m³)', 
                'Ozone (µg/m³)'
            ]
            # Resample and forward-fill as before
            df_new = df_new[pollutants_cols].resample('15T').mean().ffill()
            
            # Scale using the SAME scaler from training
            scaled_new = scaler.transform(df_new)
            
            # Create sliding window for new data
            # Make sure you use the same n_steps = 4
            n_steps = 4
            X_new, y_new = [], []
            
            # We want the index of "pollutant" in pollutants_cols for the target
            target_idx = pollutants_cols.index(pollutant)
            
            for i in range(len(scaled_new) - n_steps):
                X_new.append(scaled_new[i : i + n_steps, :])
                y_new.append(scaled_new[i + n_steps, target_idx])
            
            X_new = np.array(X_new)
            y_new = np.array(y_new)
            
            if len(X_new) == 0:
                # If the new CSV is too small or empty after resampling, skip
                continue
            
            # Evaluate
            mse, mae = model.evaluate(X_new, y_new, verbose=0)
            
            # Collect results
            evaluation_records.append({
                'timestamp': datetime.now().isoformat(),
                'pollutant': pollutant,
                'new_csv': csv_file,
                'mse': mse,
                'mae': mae
            })
            print(f"[{pollutant}] {csv_file} => MSE: {mse:.4f}, MAE: {mae:.4f}")
    
    # Step E: Save all evaluation results to a CSV
    df_eval = pd.DataFrame(evaluation_records, columns=columns)
    df_eval.to_csv(output_file, index=False)
    print(f"\nSaved evaluation results to: {output_file}")


# -------------------------------------------------
# Usage:
# -------------------------------------------------
# Make sure you have already:
#   1. Run the main training experiment: run_experiment(DATA_PATH)
#   2. Verified "model_results.csv" is present
# Then call:

evaluate_on_new_data(
    results_file='model_results.csv',
    new_data_folder='Non_null_datasets',
    output_file='new_data_evaluation.csv'
)
# This will produce new_data_evaluation.csv with MSE/MAE for each new CSV and each pollutant.


C:\Users\DELL\AppData\Local\Temp\ipykernel_420\2707660949.py:16: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df = df[pollutants].resample('15T').mean().ffill()
c:\Users\DELL\Documents\Amrita\4th year\AirQuality_FederatedLearning\venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
C:\Users\DELL\AppData\Local\Temp\ipykernel_420\109361474.py:112: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_new = df_new[pollutants_cols].resample('15T').mean().ffill()


[PM2.5 (µg/m³)] preprocessed_Raw_data_15Min_2024_site_1431_Patparganj_Delhi_DPCC_15Min.csv => MSE: 0.0014, MAE: 0.0174


C:\Users\DELL\AppData\Local\Temp\ipykernel_420\109361474.py:112: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_new = df_new[pollutants_cols].resample('15T').mean().ffill()


[PM2.5 (µg/m³)] preprocessed_Raw_data_15Min_2024_site_1434_Wazirpur_Delhi_DPCC_15Min.csv => MSE: 0.0018, MAE: 0.0232


C:\Users\DELL\AppData\Local\Temp\ipykernel_420\109361474.py:112: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_new = df_new[pollutants_cols].resample('15T').mean().ffill()


...

[NO (µg/m³)] preprocessed_Raw_data_15Min_2024_site_1561_Mundka_Delhi_DPCC_15Min.csv => MSE: 0.0029, MAE: 0.0173


C:\Users\DELL\AppData\Local\Temp\ipykernel_420\109361474.py:112: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_new = df_new[pollutants_cols].resample('15T').mean().ffill()


[NO (µg/m³)] preprocessed_Raw_data_15Min_2024_site_1563_Pusa_Delhi_DPCC_15Min.csv => MSE: 0.0197, MAE: 0.0580


C:\Users\DELL\AppData\Local\Temp\ipykernel_420\109361474.py:112: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_new = df_new[pollutants_cols].resample('15T').mean().ffill()


KeyboardInterrupt: 

## 14th feb , 


In [ ]:
# Cell 1

import numpy as np
import pandas as pd
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime
import math

# Set random seed for reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Approximate lat/lon for your Delhi localities
# (Coordinates are approximate, found via online sources; for demonstration only)
city_coords_example = {
    "Patparganj_Delhi": (28.6352, 77.3057),
    "Wazirpur_Delhi": (28.6933, 77.1546),
    "Pusa_Delhi": (28.6394, 77.1674),
    "Mundka_Delhi": (28.6848, 77.0351),
    "Alipur_Delhi": (28.7981, 77.1476)
}

# List of pollutants used in your code
POLLUTANTS = [
    'PM2.5 (µg/m³)', 'PM10 (µg/m³)', 'NO (µg/m³)',
    'NO2 (µg/m³)', 'SO2 (µg/m³)', 'CO (mg/m³)',
    'Ozone (µg/m³)'
]


In [ ]:
# Cell 2

# Approximate pairwise distances (in km) among the five localities
# These can be used if you want to do lookups directly rather than computing on the fly.
distance_dict_example = {
    ("Patparganj_Delhi", "Wazirpur_Delhi"): 15.0,
    ("Patparganj_Delhi", "Pusa_Delhi"): 13.0,
    ("Patparganj_Delhi", "Mundka_Delhi"): 27.0,
    ("Patparganj_Delhi", "Alipur_Delhi"): 34.0,

    ("Wazirpur_Delhi", "Patparganj_Delhi"): 15.0,  # symmetrical
    ("Wazirpur_Delhi", "Pusa_Delhi"): 7.0,
    ("Wazirpur_Delhi", "Mundka_Delhi"): 16.0,
    ("Wazirpur_Delhi", "Alipur_Delhi"): 17.0,

    ("Pusa_Delhi", "Patparganj_Delhi"): 13.0,
    ("Pusa_Delhi", "Wazirpur_Delhi"): 7.0,
    ("Pusa_Delhi", "Mundka_Delhi"): 17.0,
    ("Pusa_Delhi", "Alipur_Delhi"): 21.0,

    ("Mundka_Delhi", "Patparganj_Delhi"): 27.0,
    ("Mundka_Delhi", "Wazirpur_Delhi"): 16.0,
    ("Mundka_Delhi", "Pusa_Delhi"): 17.0,
    ("Mundka_Delhi", "Alipur_Delhi"): 23.0,

    ("Alipur_Delhi", "Patparganj_Delhi"): 34.0,
    ("Alipur_Delhi", "Wazirpur_Delhi"): 17.0,
    ("Alipur_Delhi", "Pusa_Delhi"): 21.0,
    ("Alipur_Delhi", "Mundka_Delhi"): 23.0,
}

def knn_filter_cities_by_distance(ref_city, distance_dict, max_k=2, corr_threshold=0.0):
    """
    A simple function that finds the K= max_k nearest neighbors of 'ref_city'
    based on the provided distance_dict_example.
    'corr_threshold' can be used if you have correlation-based filtering logic.
    """
    dist_list = []
    for (cityA, cityB), dist in distance_dict.items():
        if cityA == ref_city:
            # If correlation is needed, you'd check it here. For demonstration, we skip that.
            if dist >= 0:  # no correlation check here
                dist_list.append((cityB, dist))
    
    # Sort by distance ascending
    dist_list.sort(key=lambda x: x[1])
    # Return up to max_k cities
    selected_neighbors = [city for city, d in dist_list[:max_k]]
    return selected_neighbors

# Example usage
neighbors_for_patparganj = knn_filter_cities_by_distance("Patparganj_Delhi", distance_dict_example, max_k=2)
print("Neighbors for Patparganj_Delhi:", neighbors_for_patparganj)


Neighbors for Patparganj_Delhi: ['Pusa_Delhi', 'Wazirpur_Delhi']


In [ ]:
# Cell 3

def load_and_preprocess_data(
    file_path,
    target_pollutant,
    neighbors_data_paths=None,
    n_steps=4,
    test_size=0.2
):
    """
    Loads dataset for the main city from 'file_path'.
    Optionally merges data from neighbor cities in 'neighbors_data_paths'.
    Creates sliding window dataset for time-series forecasting.
    """
    # 1) Load primary city data
    df_main = pd.read_csv(file_path, parse_dates=['Timestamp'], index_col='Timestamp')
    
    # Pollutants of interest
    pollutants = [
        'PM2.5 (µg/m³)', 'PM10 (µg/m³)', 'NO (µg/m³)',
        'NO2 (µg/m³)', 'SO2 (µg/m³)', 'CO (mg/m³)',
        'Ozone (µg/m³)'
    ]
    
    # Resample to 15-min intervals if needed, then forward-fill
    df_main = df_main[pollutants].resample('15T').mean().ffill()
    
    # 2) Merge neighbors (optional)
    if neighbors_data_paths is not None:
        for n_path in neighbors_data_paths:
            df_n = pd.read_csv(n_path, parse_dates=['Timestamp'], index_col='Timestamp')
            df_n = df_n[pollutants].resample('15T').mean().ffill()
            
            # Rename neighbor columns to avoid collision
            base_name = os.path.splitext(os.path.basename(n_path))[0]
            rename_cols = {col: f"{base_name}_{col}" for col in df_n.columns}
            df_n.rename(columns=rename_cols, inplace=True)
            
            # Join with main
            df_main = df_main.join(df_n, how='outer')
        # forward-fill any new gaps
        df_main.ffill(inplace=True)

    # 3) Scale data
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(df_main.values)
    scaled_df = pd.DataFrame(scaled_data, index=df_main.index, columns=df_main.columns)
    
    # 4) Create time-series windows
    def create_dataset(data, n_steps, target_col_idx):
        X, y = [], []
        for i in range(len(data) - n_steps):
            X.append(data[i:i+n_steps, :])
            y.append(data[i+n_steps, target_col_idx])
        return np.array(X), np.array(y)
    
    # Identify the target column index
    target_idx = list(scaled_df.columns).index(target_pollutant)
    
    data_array = scaled_df.values
    X_all, y_all = create_dataset(data_array, n_steps, target_idx)
    
    # 5) Train-test split (no shuffle for time-series)
    n_test = int(len(X_all) * test_size)
    X_train, X_test = X_all[:-n_test], X_all[-n_test:]
    y_train, y_test = y_all[:-n_test], y_all[-n_test:]
    
    return (X_train, y_train), (X_test, y_test), scaler, scaled_df.shape[1]

# Example usage:
DATA_PATH = "Non_Null_Datasets/preprocessed_Raw_data_15Min_2024_site_5024_Alipur_Delhi_DPCC_15Min.csv"

# Suppose you found 2 nearest neighbors for Alipur_Delhi, and those CSVs are:
neighbor_paths = [
    "Non_Null_Datasets\preprocessed_Raw_data_15Min_2024_site_1563_Pusa_Delhi_DPCC_15Min.csv",
    "Non_Null_Datasets\preprocessed_Raw_data_15Min_2024_site_1434_Wazirpur_Delhi_DPCC_15Min.csv"
]

(X_train, y_train), (X_test, y_test), scaler, n_features = load_and_preprocess_data(
    file_path=DATA_PATH,
    target_pollutant='PM2.5 (µg/m³)',
    neighbors_data_paths=neighbor_paths,  # or None if no neighbors
    n_steps=4,
    test_size=0.2
)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("n_features after merging neighbors:", n_features)


C:\Users\DELL\AppData\Local\Temp\ipykernel_420\3589761165.py:26: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_main = df_main[pollutants].resample('15T').mean().ffill()
C:\Users\DELL\AppData\Local\Temp\ipykernel_420\3589761165.py:32: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_n = df_n[pollutants].resample('15T').mean().ffill()


X_train shape: (22576, 4, 21)
y_train shape: (22576,)
n_features after merging neighbors: 21


C:\Users\DELL\AppData\Local\Temp\ipykernel_420\3589761165.py:32: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_n = df_n[pollutants].resample('15T').mean().ffill()


In [ ]:
# Cell 4 (Revised)
# Import K from keras for certain operations
from tensorflow.keras import backend as K

def channel_attention(input_tensor, ratio=8):
    """
    Channel Attention:
      - Uses GlobalAveragePooling2D and GlobalMaxPooling2D to generate descriptors of shape (batch, channels)
      - Reshapes to (batch, 1, 1, channels)
      - Passes them through an MLP (2 Dense layers) to get a channel attention map
      - Sigmoid + Multiply with original feature map
    """
    channel = int(input_tensor.shape[-1])  # Convert to int

    # Global Average Pooling -> shape (batch, channels)
    avg_pool = layers.GlobalAveragePooling2D()(input_tensor)
    # Global Max Pooling -> shape (batch, channels)
    max_pool = layers.GlobalMaxPooling2D()(input_tensor)

    # Reshape to (batch,1,1,channels)
    avg_pool = layers.Reshape((1, 1, channel))(avg_pool)
    max_pool = layers.Reshape((1, 1, channel))(max_pool)

    hidden_units = max(channel // ratio, 1)
    mlp = tf.keras.Sequential([
        layers.Dense(hidden_units, activation='relu'),
        layers.Dense(channel)
    ])

    avg_out = mlp(avg_pool)  # shape (batch,1,1,channel)
    max_out = mlp(max_pool)  # shape (batch,1,1,channel)

    # Add & sigmoid
    out = layers.Add()([avg_out, max_out])  # (batch,1,1,channel)
    out = layers.Activation('sigmoid')(out)

    # Broadcast multiply: (batch,H,W,channel) * (batch,1,1,channel)
    return layers.Multiply()([input_tensor, out])


def spatial_attention(input_tensor, kernel_size=7):
    """
    Spatial Attention:
      - Computes mean and max along the channel axis
      - Concatenates them and applies Conv2D (filter=1, sigmoid)
      - Multiplies the resulting map with the input
    """
    # Compute mean along channels
    avg_pool = layers.Lambda(lambda x: K.mean(x, axis=3, keepdims=True))(input_tensor)
    # Compute max along channels
    max_pool = layers.Lambda(lambda x: K.max(x, axis=3, keepdims=True))(input_tensor)

    # Concatenate along channel axis -> shape (batch, H, W, 2)
    concat = layers.Concatenate(axis=3)([avg_pool, max_pool])

    # 1x Conv -> 1 filter, sigmoid activation
    attn_map = layers.Conv2D(filters=1, kernel_size=kernel_size, padding='same', activation='sigmoid')(concat)

    # Multiply input by attention map
    return layers.Multiply()([input_tensor, attn_map])


def spatio_temporal_attention(input_tensor):
    """
    Combined channel + spatial attention:
      1) Channel attention
      2) Spatial attention
    """
    x = channel_attention(input_tensor)
    x = spatial_attention(x)
    return x


In [ ]:
# Cell 5 (Revised)
def STA_ResidualBlock(inputs, filters=32, kernel_size=3):
    """
    A residual block that integrates the spatio-temporal attention (STA):
      1) Conv -> BN -> ReLU
      2) spatio_temporal_attention
      3) Conv -> BN
      4) Residual Add -> ReLU
    """
    x = layers.Conv2D(filters, kernel_size, padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    
    # Spatio-temporal attention
    x = spatio_temporal_attention(x)
    
    x = layers.Conv2D(filters, kernel_size, padding='same', activation=None)(x)
    x = layers.BatchNormalization()(x)
    
    # If channel mismatch, project input to match dimension
    if inputs.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, kernel_size=1, padding='same')(inputs)
    else:
        shortcut = inputs
    
    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)
    
    return x


In [ ]:
# Cell 6 (Revised)

def build_ksc_convlstm(
    input_shape,
    num_res_blocks=1,
    filters=32,
    kernel_size=3,
    lstm_filters=64,
    output_units=1
):
    """
    KSC-ConvLSTM model architecture:
    - Input shape: (n_steps, n_features)
    - One or more STA_ResidualBlock
    - Reshape -> ConvLSTM2D -> Flatten -> Dense
    """
    inputs = layers.Input(shape=input_shape)  # e.g., (4, total_features)
    
    # Instead of tf.expand_dims, use a Keras Reshape layer
    x = layers.Reshape((input_shape[0], input_shape[1], 1))(inputs)
    
    # STA-Residual blocks
    for _ in range(num_res_blocks):
        x = STA_ResidualBlock(x, filters=filters, kernel_size=kernel_size)
    
    # Reshape for ConvLSTM
    # ConvLSTM expects (batch, time, rows, cols, channels)
    # We'll interpret: time = n_steps, rows=1, cols=n_features, channels=filters
    x = layers.Reshape((input_shape[0], 1, input_shape[1], filters))(x)
    
    # ConvLSTM2D
    x = layers.ConvLSTM2D(
        filters=lstm_filters,
        kernel_size=(1,3),
        padding='same',
        return_sequences=False
    )(x)
    
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation='relu')(x)
    outputs = layers.Dense(output_units)(x)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="KSC_ConvLSTM")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='mse',
        metrics=['mae']
    )
    
    return model


In [ ]:
# Cell 7

def train_ksc_convlstm(
    X_train, y_train,
    X_val, y_val,
    n_features,
    epochs=50,
    batch_size=128
):
    """
    Trains the KSC-ConvLSTM model using given train/val data.
    """
    model = build_ksc_convlstm(
        input_shape=(X_train.shape[1], n_features),
        num_res_blocks=1,     # or more
        filters=32,
        kernel_size=3,
        lstm_filters=64,
        output_units=1
    )
    
    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True
    )
    
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=1
    )
    
    return model, history

# Example usage:
val_ratio = 0.2
n_val = int(len(X_train) * val_ratio)
X_val, y_val = X_train[-n_val:], y_train[-n_val:]
X_train2, y_train2 = X_train[:-n_val], y_train[:-n_val]

model, history = train_ksc_convlstm(
    X_train2, y_train2,
    X_val, y_val,
    n_features=n_features,
    epochs=50,
    batch_size=128
)



Epoch 1/50
142/142 ━━━━━━━━━━━━━━━━━━━━ 11s 49ms/step - loss: 0.0109 - mae: 0.0695 - val_loss: 0.0057 - val_mae: 0.0617
Epoch 2/50
142/142 ━━━━━━━━━━━━━━━━━━━━ 7s 52ms/step - loss: 0.0014 - mae: 0.0272 - val_loss: 0.0050 - val_mae: 0.0579
Epoch 3/50
142/142 ━━━━━━━━━━━━━━━━━━━━ 7s 47ms/step - loss: 0.0011 - mae: 0.0235 - val_loss: 0.0038 - val_mae: 0.0525
Epoch 4/50
142/142 ━━━━━━━━━━━━━━━━━━━━ 6s 44ms/step - loss: 9.6949e-04 - mae: 0.0220 - val_loss: 0.0021 - val_mae: 0.0401
Epoch 5/50
142/142 ━━━━━━━━━━━━━━━━━━━━ 6s 41ms/step - loss: 8.7254e-04 - mae: 0.0208 - val_loss: 9.8537e-04 - val_mae: 0.0265
Epoch 6/50
142/142 ━━━━━━━━━━━━━━━━━━━━ 6s 39ms/step - loss: 8.1259e-04 - mae: 0.0199 - val_loss: 6.0452e-04 - val_mae: 0.0200
Epoch 7/50
142/142 ━━━━━━━━━━━━━━━━━━━━ 6s 41ms/step - loss: 7.6974e-04 - mae: 0.0193 - val_loss: 4.8234e-04 - val_mae: 0.0177
Epoch 8/50
142/142 ━━━━━━━━━━━━━━━━━━━━ 7s 47ms/step - loss: 7.3924e-04 - mae: 0.0188 - val_loss: 4.1480e-04 - val_mae: 0.0163
Epoch 9/50

In [ ]:
# Cell 8

# Evaluate on the test set
mse, mae = model.evaluate(X_test, y_test, verbose=1)
print(f"Test MSE: {mse:.4f}, MAE: {mae:.4f}")

y_pred = model.predict(X_test)

def inverse_transform_predictions(y_scaled, scaler, target_col_name):
    """
    Inverse-transform a 1D array of scaled predictions, placing them back
    into a zero array, then calling scaler.inverse_transform.
    """
    n = len(y_scaled)
    n_feat = scaler.scale_.shape[0]
    
    # Build a placeholder
    place = np.zeros((n, n_feat))
    # Use a known source of columns or pollutant list
    col_idx = POLLUTANTS.index(target_col_name)
    place[:, col_idx] = y_scaled
    
    # Invert transform
    place_inv = scaler.inverse_transform(place)
    return place_inv[:, col_idx]

target_col = 'PM2.5 (µg/m³)'
y_test_inv = inverse_transform_predictions(y_test,  scaler, target_col)
y_pred_inv = inverse_transform_predictions(y_pred.flatten(), scaler, target_col)

rmse_orig = np.sqrt(np.mean((y_test_inv - y_pred_inv)**2))
mae_orig  = np.mean(np.abs(y_test_inv - y_pred_inv))
print(f"Test RMSE (original scale): {rmse_orig:.2f}")
print(f"Test MAE  (original scale): {mae_orig:.2f}")


177/177 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 2.9716e-04 - mae: 0.0127
Test MSE: 0.0007, MAE: 0.0159
177/177 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
Test RMSE (original scale): 12.05
Test MAE  (original scale): 7.23


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Load your data
data_path = DATA_PATH  # Update this to the path of your dataset
df = pd.read_csv(data_path, parse_dates=['Timestamp'], index_col='Timestamp')

# Fill missing values if any
df.fillna(method='ffill', inplace=True)  # Forward fill
df.fillna(method='bfill', inplace=True)  # Backward fill for initial missing values

# Define pollutants and features
pollutants = [
    'PM2.5 (µg/m³)', 'PM10 (µg/m³)', 'NO (µg/m³)', 'NO2 (µg/m³)', 
    'SO2 (µg/m³)', 'CO (mg/m³)', 'Ozone (µg/m³)'
]

# Extract date-time features
df['hour'] = df.index.hour
df['day'] = df.index.day
df['dayofweek'] = df.index.dayofweek
df['month'] = df.index.month

# Scale the features
scaler = MinMaxScaler()
scaled_columns = ['hour', 'day', 'dayofweek', 'month'] + pollutants + df.columns.difference(pollutants).tolist()
df[scaled_columns] = scaler.fit_transform(df[scaled_columns])

# Define the results list to collect all metrics
results = []

# Loop over each pollutant
for pollutant in pollutants:
    target = df[pollutant]
    features = df.drop(pollutants, axis=1)  # Use only date-time and additional features

    # Split the dataset
    X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42, shuffle=False)

    # Random Forest Model
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_model.fit(X_train, y_train)
    rf_predictions = rf_model.predict(X_test)
    rf_mse = mean_squared_error(y_test, rf_predictions)
    rf_mae = mean_absolute_error(y_test, rf_predictions)

    # XGBoost Model
    xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
    xgb_model.fit(X_train, y_train)
    xgb_predictions = xgb_model.predict(X_test)
    xgb_mse = mean_squared_error(y_test, xgb_predictions)
    xgb_mae = mean_absolute_error(y_test, xgb_predictions)

    results.append({
        'Pollutant': pollutant,
        'RF_MSE': rf_mse,
        'RF_MAE': rf_mae,
        'XGB_MSE': xgb_mse,
        'XGB_MAE': xgb_mae
    })

# Convert results to DataFrame for better visualization
results_df = pd.DataFrame(results)
print(results_df)


C:\Users\DELL\AppData\Local\Temp\ipykernel_420\898987835.py:14: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)  # Forward fill
C:\Users\DELL\AppData\Local\Temp\ipykernel_420\898987835.py:15: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='bfill', inplace=True)  # Backward fill for initial missing values
c:\Users\DELL\Documents\Amrita\4th year\AirQuality_FederatedLearning\venv\Lib\site-packages\sklearn\utils\_array_api.py:769: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmin(X, axis=axis))
c:\Users\DELL\Documents\Amrita\4th year\AirQuality_FederatedLearning\venv\Lib\site-packages\sklearn\utils\_array_api.py:786: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmax(X, axis=axis))


       Pollutant    RF_MSE    RF_MAE   XGB_MSE   XGB_MAE
0  PM2.5 (µg/m³)  0.009973  0.065659  0.009605  0.063890
1   PM10 (µg/m³)  0.014904  0.082965  0.014632  0.085489
2     NO (µg/m³)  0.000131  0.005613  0.000133  0.005054
3    NO2 (µg/m³)  0.001674  0.022866  0.000898  0.018371
4    SO2 (µg/m³)  0.000660  0.015359  0.000615  0.016843
5     CO (mg/m³)  0.003274  0.046715  0.002970  0.043268
6  Ozone (µg/m³)  0.018069  0.093669  0.018694  0.091105


## without SCALING

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler

# Assuming models are already trained and are named rf_model and xgb_model
# If models are not trained, you can serialize them after training using joblib or pickle and then load them

# Load the models if not in memory (Uncomment if models are saved)
# import joblib
# rf_model = joblib.load('rf_model.pkl')
# xgb_model = joblib.load('xgb_model.pkl')

# New datasets paths
dataset_paths =[
    'Non_Null_Datasets/preprocessed_Raw_data_15Min_2024_site_1431_Patparganj_Delhi_DPCC_15Min.csv',
    'Non_Null_Datasets/preprocessed_Raw_data_15Min_2024_site_1561_Mundka_Delhi_DPCC_15Min.csv',
    'Non_Null_Datasets/preprocessed_Raw_data_15Min_2024_site_1434_Wazirpur_Delhi_DPCC_15Min.csv',
    'Non_Null_Datasets/preprocessed_Raw_data_15Min_2024_site_1563_Pusa_Delhi_DPCC_15Min.csv',
    'Non_Null_Datasets/preprocessed_Raw_data_15Min_2024_site_5024_Alipur_Delhi_DPCC_15Min.csv'
]
# Update these paths

# Function to preprocess data (similar to your training preprocessing)
def preprocess_data(data_path):
    df = pd.read_csv(data_path, parse_dates=['Timestamp'], index_col='Timestamp')
    df.fillna(method='ffill', inplace=True)
    df.fillna(method='bfill', inplace=True)
    
    # Extract date-time features
    df['hour'] = df.index.hour
    df['day'] = df.index.day
    df['dayofweek'] = df.index.dayofweek
    df['month'] = df.index.month

    # Scale the features
    scaler = MinMaxScaler()
    scaled_columns = ['hour', 'day', 'dayofweek', 'month'] + pollutants
    df[scaled_columns] = df[scaled_columns]
    
    return df

# List to store results
results = []

# Process each dataset
for path in dataset_paths:
    df = preprocess_data(path)
    
    for pollutant in pollutants:
        features = df.drop(pollutants, axis=1)  # Using the same features as for training
        target = df[pollutant]

        # Predict using Random Forest
        rf_predictions = rf_model.predict(features)
        rf_mse = mean_squared_error(target, rf_predictions)
        rf_mae = mean_absolute_error(target, rf_predictions)

        # Predict using XGBoost
        xgb_predictions = xgb_model.predict(features)
        xgb_mse = mean_squared_error(target, xgb_predictions)
        xgb_mae = mean_absolute_error(target, xgb_predictions)

        results.append({
            'Dataset': path,
            'Pollutant': pollutant,
            'RF_MSE': rf_mse,
            'RF_MAE': rf_mae,
            'XGB_MSE': xgb_mse,
            'XGB_MAE': xgb_mae
        })

# Convert results to DataFrame for better visualization
results_df = pd.DataFrame(results)
print(results_df)


C:\Users\DELL\AppData\Local\Temp\ipykernel_420\1816717953.py:29: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)
C:\Users\DELL\AppData\Local\Temp\ipykernel_420\1816717953.py:30: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='bfill', inplace=True)
C:\Users\DELL\AppData\Local\Temp\ipykernel_420\1816717953.py:29: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)
C:\Users\DELL\AppData\Local\Temp\ipykernel_420\1816717953.py:30: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='bfill', inplace=True)
C:\Users\DELL\AppData\Local\

                                              Dataset      Pollutant  \
0   Non_Null_Datasets/preprocessed_Raw_data_15Min_...  PM2.5 (µg/m³)   
1   Non_Null_Datasets/preprocessed_Raw_data_15Min_...   PM10 (µg/m³)   
2   Non_Null_Datasets/preprocessed_Raw_data_15Min_...     NO (µg/m³)   
3   Non_Null_Datasets/preprocessed_Raw_data_15Min_...    NO2 (µg/m³)   
4   Non_Null_Datasets/preprocessed_Raw_data_15Min_...    SO2 (µg/m³)   
5   Non_Null_Datasets/preprocessed_Raw_data_15Min_...     CO (mg/m³)   
6   Non_Null_Datasets/preprocessed_Raw_data_15Min_...  Ozone (µg/m³)   
7   Non_Null_Datasets/preprocessed_Raw_data_15Min_...  PM2.5 (µg/m³)   
8   Non_Null_Datasets/preprocessed_Raw_data_15Min_...   PM10 (µg/m³)   
9   Non_Null_Datasets/preprocessed_Raw_data_15Min_...     NO (µg/m³)   
10  Non_Null_Datasets/preprocessed_Raw_data_15Min_...    NO2 (µg/m³)   
11  Non_Null_Datasets/preprocessed_Raw_data_15Min_...    SO2 (µg/m³)   
12  Non_Null_Datasets/preprocessed_Raw_data_15Min_...     CO (mg

In [47]:
# %%
# Single Cell: Combined Model Training and Evaluation (Deep Learning and ML models without scaling)

import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from tensorflow.keras.layers import Input, Dense, LSTM, Dropout, Flatten, Bidirectional, Reshape, ConvLSTM2D
from tensorflow.keras.models import Sequential
from datetime import datetime

# Setup
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Load Data
data_path = 'Non_Null_Datasets/preprocessed_Raw_data_15Min_2024_site_5024_Alipur_Delhi_DPCC_15Min.csv'
df = pd.read_csv(data_path, parse_dates=['Timestamp'], index_col='Timestamp')
df.fillna(method='ffill', inplace=True)
df.fillna(method='bfill', inplace=True)

# Define pollutants
pollutants = [
    'PM2.5 (µg/m³)', 'PM10 (µg/m³)', 'NO (µg/m³)',
    'NO2 (µg/m³)', 'SO2 (µg/m³)', 'CO (mg/m³)', 
    'Ozone (µg/m³)'
]

# Function to create a dataset with n_steps
def create_dataset(X, y, time_steps=1):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X.iloc[i:(i + time_steps)].values)
        ys.append(y.iloc[i + time_steps])
    return np.array(Xs), np.array(ys)

# Prepare data for each pollutant
results = []
for pollutant in pollutants:
    target = df[pollutant]
    features = df.drop(pollutants, axis=1)  # Exclude other pollutants to avoid leakage

    # Creating timestep data
    time_steps = 4
    X, y = create_dataset(df[features.columns], target, time_steps)

    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, shuffle=False)

    # Define the model architecture (MLP-LSTM Hybrid without scaling)
    model = Sequential([
        Flatten(input_shape=(X_train.shape[1], X_train.shape[2])),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Reshape((1, -1)),
        LSTM(128, return_sequences=False),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(loss='mean_squared_error', optimizer='adam', metrics=['mae'])

    # Train the model
    model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=0, validation_split=0.2)

    # Evaluate the model
    mse, mae = model.evaluate(X_test, y_test, verbose=0)
    results.append({'Model': 'Deep Learning', 'Pollutant': pollutant, 'MSE': mse, 'MAE': mae})

    # Train Random Forest and XGBoost models
    rf_model = RandomForestRegressor(n_estimators=100, random_state=SEED)
    rf_model.fit(X_train.reshape(X_train.shape[0], -1), y_train)  # Reshape for sklearn
    rf_predictions = rf_model.predict(X_test.reshape(X_test.shape[0], -1))

    xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=SEED)
    xgb_model.fit(X_train.reshape(X_train.shape[0], -1), y_train)
    xgb_predictions = xgb_model.predict(X_test.reshape(X_test.shape[0], -1))

    # Evaluate ML models
    rf_mse = mean_squared_error(y_test, rf_predictions)
    rf_mae = mean_absolute_error(y_test, rf_predictions)
    xgb_mse = mean_squared_error(y_test, xgb_predictions)
    xgb_mae = mean_absolute_error(y_test, xgb_predictions)

    results.append({'Model': 'Random Forest', 'Pollutant': pollutant, 'MSE': rf_mse, 'MAE': rf_mae})
    results.append({'Model': 'XGBoost', 'Pollutant': pollutant, 'MSE': xgb_mse, 'MAE': xgb_mae})

# Print results
results_df = pd.DataFrame(results)
print(results_df)

# %%


C:\Users\DELL\AppData\Local\Temp\ipykernel_420\608445428.py:23: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)
C:\Users\DELL\AppData\Local\Temp\ipykernel_420\608445428.py:24: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='bfill', inplace=True)
c:\Users\DELL\Documents\Amrita\4th year\AirQuality_FederatedLearning\venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Users\DELL\Documents\Amrita\4th year\AirQuality_FederatedLearning\venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_

            Model      Pollutant           MSE         MAE
0   Deep Learning  PM2.5 (µg/m³)   3535.826416   51.307419
1   Random Forest  PM2.5 (µg/m³)   1440.438089   24.396494
2         XGBoost  PM2.5 (µg/m³)   1531.424137   25.854488
3   Deep Learning   PM10 (µg/m³)  16455.423828  111.369431
4   Random Forest   PM10 (µg/m³)   5975.513351   56.045539
5         XGBoost   PM10 (µg/m³)   6553.149185   59.534413
6   Deep Learning     NO (µg/m³)    166.772797    6.098637
7   Random Forest     NO (µg/m³)     35.804001    2.560584
8         XGBoost     NO (µg/m³)     51.457687    2.499530
9   Deep Learning    NO2 (µg/m³)    233.921265   12.548366
10  Random Forest    NO2 (µg/m³)     30.636055    3.504208
11        XGBoost    NO2 (µg/m³)     29.244756    3.500727
12  Deep Learning    SO2 (µg/m³)     85.902855    8.707210
13  Random Forest    SO2 (µg/m³)     43.740676    4.124688
14        XGBoost    SO2 (µg/m³)     51.780864    4.295913
15  Deep Learning     CO (mg/m³)      0.217859    0.4338

In [48]:
# %%
# Single Cell: Custom Model Training and Evaluation for Each Pollutant

import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from tensorflow.keras.layers import Input, Dense, LSTM, Dropout, Flatten, Bidirectional, Reshape, ConvLSTM2D
from tensorflow.keras.models import Sequential
from datetime import datetime

# Setup
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Load Data
data_path = 'Non_Null_Datasets/preprocessed_Raw_data_15Min_2024_site_5024_Alipur_Delhi_DPCC_15Min.csv'
df = pd.read_csv(data_path, parse_dates=['Timestamp'], index_col='Timestamp')
df.fillna(method='ffill', inplace=True)
df.fillna(method='bfill', inplace=True)

# Define pollutants and configurations
pollutants = {
    'PM2.5 (µg/m³)': ([128], 2, 64, False),
    'PM10 (µg/m³)': ([128], 2, 64, False),
    'NO (µg/m³)': ([256, 128], 1, 256, True),
    'NO2 (µg/m³)': ([128, 64, 32], 1, 64, True),
    'SO2 (µg/m³)': ([128], 2, 64, False),
    'CO (mg/m³)': ([128], 2, 64, False),
    'Ozone (µg/m³)': ([128], 2, 64, False)
}

# Function to create a dataset with n_steps
def create_dataset(X, y, time_steps=1):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X.iloc[i:(i + time_steps)].values)
        ys.append(y.iloc[i + time_steps])
    return np.array(Xs), np.array(ys)

# Prepare data and train models for each pollutant
results = []
time_steps = 4
for pollutant, config in pollutants.items():
    target = df[pollutant]
    features = df.drop(list(pollutants.keys()), axis=1)  # Exclude other pollutants to avoid leakage

    X, y = create_dataset(df[features.columns], target, time_steps)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, shuffle=False)

    # Build MLP-LSTM model according to the pollutant-specific configuration
    model = Sequential()
    model.add(Flatten(input_shape=(X_train.shape[1], X_train.shape[2])))
    for units in config[0]:
        model.add(Dense(units, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Reshape((1, -1)))
    if config[3]:  # BiLSTM
        model.add(Bidirectional(LSTM(config[2], return_sequences=config[1] > 1)))
    else:
        model.add(LSTM(config[2], return_sequences=config[1] > 1))
    if config[1] > 1:  # Additional LSTM layer if needed
        model.add(LSTM(config[2], return_sequences=False))
    model.add(Dropout(0.2))
    model.add(Dense(1))
    model.compile(loss='mean_squared_error', optimizer='adam', metrics=['mae'])

    # Train the deep learning model
    model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=0, validation_split=0.2)
    mse, mae = model.evaluate(X_test, y_test, verbose=0)
    results.append({'Model': 'Deep Learning', 'Pollutant': pollutant, 'MSE': mse, 'MAE': mae})

    # Train Random Forest and XGBoost models
    rf_model = RandomForestRegressor(n_estimators=100, random_state=SEED)
    rf_model.fit(X_train.reshape(X_train.shape[0], -1), y_train)
    rf_predictions = rf_model.predict(X_test.reshape(X_test.shape[0], -1))
    rf_mse = mean_squared_error(y_test, rf_predictions)
    rf_mae = mean_absolute_error(y_test, rf_predictions)
    results.append({'Model': 'Random Forest', 'Pollutant': pollutant, 'MSE': rf_mse, 'MAE': rf_mae})

    xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=SEED)
    xgb_model.fit(X_train.reshape(X_train.shape[0], -1), y_train)
    xgb_predictions = xgb_model.predict(X_test.reshape(X_test.shape[0], -1))
    xgb_mse = mean_squared_error(y_test, xgb_predictions)
    xgb_mae = mean_absolute_error(y_test, xgb_predictions)
    results.append({'Model': 'XGBoost', 'Pollutant': pollutant, 'MSE': xgb_mse, 'MAE': xgb_mae})

# Print results
results_df = pd.DataFrame(results)
print(results_df)

# %%


C:\Users\DELL\AppData\Local\Temp\ipykernel_420\1896194805.py:23: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)
C:\Users\DELL\AppData\Local\Temp\ipykernel_420\1896194805.py:24: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='bfill', inplace=True)
c:\Users\DELL\Documents\Amrita\4th year\AirQuality_FederatedLearning\venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Users\DELL\Documents\Amrita\4th year\AirQuality_FederatedLearning\venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `inpu

            Model      Pollutant           MSE         MAE
0   Deep Learning  PM2.5 (µg/m³)   3514.584717   51.081936
1   Random Forest  PM2.5 (µg/m³)   1440.438089   24.396494
2         XGBoost  PM2.5 (µg/m³)   1531.424137   25.854488
3   Deep Learning   PM10 (µg/m³)  16398.472656  111.117195
4   Random Forest   PM10 (µg/m³)   5975.513351   56.045539
5         XGBoost   PM10 (µg/m³)   6553.149185   59.534413
6   Deep Learning     NO (µg/m³)    166.697784    6.138426
7   Random Forest     NO (µg/m³)     35.804001    2.560584
8         XGBoost     NO (µg/m³)     51.457687    2.499530
9   Deep Learning    NO2 (µg/m³)    233.597916   12.533804
10  Random Forest    NO2 (µg/m³)     30.636055    3.504208
11        XGBoost    NO2 (µg/m³)     29.244756    3.500727
12  Deep Learning    SO2 (µg/m³)     84.269066    8.616967
13  Random Forest    SO2 (µg/m³)     43.740676    4.124688
14        XGBoost    SO2 (µg/m³)     51.780864    4.295913
15  Deep Learning     CO (mg/m³)      0.209939    0.4251

## proper time series with stationary data conversion

In [68]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.vector_ar.var_model import VAR
from keras.models import Sequential
from keras.layers import GRU, Dense
from keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler

import warnings
warnings.filterwarnings('ignore')

# Data Loading and Preprocessing
def load_data(file_path):
    df = pd.read_csv(file_path, parse_dates=['Timestamp'], index_col='Timestamp')
    pollutants = [
        'PM2.5 (µg/m³)', 'PM10 (µg/m³)', 'NO (µg/m³)', 
        'NO2 (µg/m³)', 'SO2 (µg/m³)', 'CO (mg/m³)', 
        'Ozone (µg/m³)'
    ]
    df = df[pollutants].resample('15T').mean().ffill()
    return df

# Model Definitions
def arima_model(train, test, order):
    model = ARIMA(train, order=order)
    model_fit = model.fit()
    predictions = model_fit.forecast(steps=len(test))
    return predictions

def sarima_model(train, test, order, seasonal_order):
    model = SARIMAX(train, order=order, seasonal_order=seasonal_order)
    model_fit = model.fit(disp=False)
    predictions = model_fit.forecast(steps=len(test))
    return predictions

def prophet_model(train, test):
    df = train.reset_index().rename(columns={'Timestamp': 'ds', train.name: 'y'})
    model = Prophet()
    model.fit(df)
    future = model.make_future_dataframe(periods=len(test), freq='15T')
    forecast = model.predict(future)
    return forecast['yhat'][-len(test):]

def gru_model(train, test):
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(train.values.reshape(-1, 1))
    
    # Create sequences
    def create_sequences(data, seq_length):
        X, y = [], []
        for i in range(len(data)-seq_length):
            X.append(data[i:i+seq_length])
            y.append(data[i+seq_length])
        return np.array(X), np.array(y)
    
    X, y = create_sequences(scaled_data, 24)  # 6-hour window
    
    model = Sequential()
    model.add(GRU(50, input_shape=(X.shape[1], X.shape[2])))
    model.add(Dense(1))
    model.compile(optimizer=Adam(0.001), loss='mse')
    model.fit(X, y, epochs=50, batch_size=32, verbose=0)
    
    # Predict
    test_scaled = scaler.transform(test.values.reshape(-1, 1))
    X_test, _ = create_sequences(test_scaled, 24)
    predictions = model.predict(X_test)
    return scaler.inverse_transform(predictions).flatten()

# Evaluation Framework
def evaluate_models(dataset_path):
    df = load_data(dataset_path)
    results = []
    
    for pollutant in df.columns:
        series = df[pollutant]
        train_size = int(len(series) * 0.8)
        train, test = series[:train_size], series[train_size:]
        
        # Try different models
        models_config = {
            'ARIMA': [{'order': (1,1,1)}, {'order': (2,1,2)}, {'order': (3,1,3)}],
            'SARIMA': [{'order': (1,1,1), 'seasonal_order': (1,1,1,24)}],
            'Prophet': [{}],
            'GRU': [{'units': 50, 'seq_length': 24}]
        }
        
        for model_name, configs in models_config.items():
            for config in configs:
                try:
                    if model_name == 'ARIMA':
                        predictions = arima_model(train, test, **config)
                    elif model_name == 'SARIMA':
                        predictions = sarima_model(train, test, **config)
                    elif model_name == 'Prophet':
                        predictions = prophet_model(train, test)
                    elif model_name == 'GRU':
                        predictions = gru_model(train, test)
                    
                    mse = mean_squared_error(test, predictions[:len(test)])
                    results.append({
                        'Pollutant': pollutant,
                        'Model': model_name,
                        'Config': str(config),
                        'MSE': mse
                    })
                except Exception as e:
                    print(f"Error with {model_name} on {pollutant}: {str(e)}")
    
    return pd.DataFrame(results)

# Run Evaluation
dataset_path = "Non_Null_Datasets\preprocessed_Raw_data_15Min_2024_site_1431_Patparganj_Delhi_DPCC_15Min.csv"

results = evaluate_models(dataset_path)
print(results)



16:20:35 - cmdstanpy - INFO - Chain [1] start processing
16:20:45 - cmdstanpy - INFO - Chain [1] done processing


176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Error with GRU on PM2.5 (µg/m³): Found input variables with inconsistent numbers of samples: [5645, 5621]


16:27:20 - cmdstanpy - INFO - Chain [1] start processing
16:27:30 - cmdstanpy - INFO - Chain [1] done processing


176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Error with GRU on PM10 (µg/m³): Found input variables with inconsistent numbers of samples: [5645, 5621]


16:32:44 - cmdstanpy - INFO - Chain [1] start processing
16:32:53 - cmdstanpy - INFO - Chain [1] done processing


...

176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Error with GRU on SO2 (µg/m³): Found input variables with inconsistent numbers of samples: [5645, 5621]


16:52:25 - cmdstanpy - INFO - Chain [1] start processing
16:52:39 - cmdstanpy - INFO - Chain [1] done processing


176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Error with GRU on CO (mg/m³): Found input variables with inconsistent numbers of samples: [5645, 5621]


16:57:54 - cmdstanpy - INFO - Chain [1] start processing
16:58:02 - cmdstanpy - INFO - Chain [1] done processing


176/176 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Error with GRU on Ozone (µg/m³): Found input variables with inconsistent numbers of samples: [5645, 5621]
        Pollutant    Model                                             Config  \
0   PM2.5 (µg/m³)    ARIMA                               {'order': (1, 1, 1)}   
1   PM2.5 (µg/m³)    ARIMA                               {'order': (2, 1, 2)}   
2   PM2.5 (µg/m³)    ARIMA                               {'order': (3, 1, 3)}   
3   PM2.5 (µg/m³)   SARIMA  {'order': (1, 1, 1), 'seasonal_order': (1, 1, ...   
4   PM2.5 (µg/m³)  Prophet                                                 {}   
5    PM10 (µg/m³)    ARIMA                               {'order': (1, 1, 1)}   
6    PM10 (µg/m³)    ARIMA                               {'order': (2, 1, 2)}   
7    PM10 (µg/m³)    ARIMA                               {'order': (3, 1, 3)}   
8    PM10 (µg/m³)   SARIMA  {'order': (1, 1, 1), 'seasonal_order': (1, 1, ...   
9    PM10 (µg/m³)  Prophet                 

In [1]:
# ============================================
# SINGLE CELL CODE: Multi-Output LSTM (Pollutants Only)
# ============================================

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# -----------------
# 1. Configuration
# -----------------
# Path to your dataset
DATA_PATH = "Non_Null_Datasets/preprocessed_Raw_data_15Min_2024_site_5024_Alipur_Delhi_DPCC_15Min.csv"

# Pollutant columns we want to forecast
POLLUTANTS = [
    'PM2.5 (µg/m³)', 'PM10 (µg/m³)', 'NO (µg/m³)', 
    'NO2 (µg/m³)', 'SO2 (µg/m³)', 'CO (mg/m³)', 
    'Ozone (µg/m³)'
]

# Number of past time steps used as input
N_STEPS = 4  # e.g., 1 hour if data is at 15-min intervals
TEST_SIZE_RATIO = 0.2
BATCH_SIZE = 32
EPOCHS = 30
SEED = 42

# Set seeds for reproducibility
np.random.seed(SEED)
tf.random.set_seed(SEED)

# -----------------
# 2. Data Loading
# -----------------
df = pd.read_csv(DATA_PATH, parse_dates=['Timestamp'], index_col='Timestamp')

# If needed, forward/backward fill missing data:
df = df.ffill().bfill()

# We only use the pollutant columns for this first model
df_pollutants = df[POLLUTANTS].copy()

# Drop any rows still containing NaNs after fill
df_pollutants.dropna(inplace=True)

# -----------------
# 3. Scaling
# -----------------
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df_pollutants.values)
scaled_df = pd.DataFrame(scaled_data, index=df_pollutants.index, columns=POLLUTANTS)

# -----------------------------
# 4. Create Multi-Output Dataset
# -----------------------------
# For each sample, we use the past N_STEPS of all pollutants as X,
# and the next step (single step ahead) of all pollutants as y.

def create_multioutput_dataset(data, n_steps):
    """
    data: 2D NumPy array of shape (num_samples, num_features)
    return: X shape (num_samples - n_steps, n_steps, num_features)
            y shape (num_samples - n_steps, num_features)
    """
    X, Y = [], []
    for i in range(len(data) - n_steps):
        X.append(data[i : i + n_steps])       # shape: (n_steps, num_features)
        Y.append(data[i + n_steps])          # shape: (num_features,)
    return np.array(X), np.array(Y)

data_array = scaled_df.values
X_all, y_all = create_multioutput_dataset(data_array, N_STEPS)

# -----------------
# 5. Train-Test Split
# -----------------
test_size = int(len(X_all) * TEST_SIZE_RATIO)

X_train, X_test = X_all[:-test_size], X_all[-test_size:]
y_train, y_test = y_all[:-test_size], y_all[-test_size:]

print("X_train shape:", X_train.shape)  # (samples, N_STEPS, #pollutants)
print("y_train shape:", y_train.shape)  # (samples, #pollutants)

# -----------------
# 6. Build the Model
# -----------------
# Hybrid MLP + LSTM example
model = tf.keras.Sequential([
    # Flatten time dimension for MLP
    tf.keras.layers.Flatten(input_shape=(N_STEPS, len(POLLUTANTS))),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    # Reshape back to (batch, timesteps, features) for LSTM
    tf.keras.layers.Reshape((1, 128)),
    tf.keras.layers.LSTM(64, return_sequences=False),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(len(POLLUTANTS))  # output = all pollutants
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

# -----------------
# 7. Train the Model
# -----------------
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

# -----------------
# 8. Evaluate
# -----------------
loss, mae = model.evaluate(X_test, y_test, verbose=0)
print(f"Test MSE: {loss:.4f}, Test MAE: {mae:.4f}")

# -----------------
# 9. Predictions & Inverse Transform
# -----------------
# Predict on test set
y_pred_scaled = model.predict(X_test)

# We need to invert scaling. Because we scaled all pollutants together,
# we can place y_pred_scaled back into the correct shape and apply inverse_transform.
def inverse_transform_multioutput(y_scaled):
    """
    y_scaled: shape (samples, #pollutants)
    Returns: shape (samples, #pollutants) in original scale
    """
    # Placeholder array for inverse
    placeholder = np.zeros((y_scaled.shape[0], len(POLLUTANTS)))
    placeholder[:] = y_scaled
    # Inverse transform
    unscaled = scaler.inverse_transform(placeholder)
    return unscaled

y_test_unscaled = inverse_transform_multioutput(y_test)
y_pred_unscaled = inverse_transform_multioutput(y_pred_scaled)

# Example: compute RMSE for each pollutant
rmse_per_pollutant = np.sqrt(np.mean((y_test_unscaled - y_pred_unscaled)**2, axis=0))

print("\nRMSE per pollutant (original units):")
for i, p in enumerate(POLLUTANTS):
    print(f"  {p}: {rmse_per_pollutant[i]:.2f}")


X_train shape: (22576, 4, 7)
y_train shape: (22576, 7)


c:\Users\DELL\Documents\Amrita\4th year\AirQuality_FederatedLearning\venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 28)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         3,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 1, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           455 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 53,575 (209.28 KB)

 Trainable params: 53,575 (209.28 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
565/565 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0079 - mae: 0.0552 - val_loss: 6.0377e-04 - val_mae: 0.0181
Epoch 2/30
565/565 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0017 - mae: 0.0278 - val_loss: 7.8458e-04 - val_mae: 0.0210
Epoch 3/30
565/565 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0014 - mae: 0.0245 - val_loss: 9.1126e-04 - val_mae: 0.0233
Epoch 4/30
565/565 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0013 - mae: 0.0232 - val_loss: 9.1349e-04 - val_mae: 0.0232
Epoch 5/30
565/565 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0012 - mae: 0.0223 - val_loss: 9.6069e-04 - val_mae: 0.0243
Epoch 6/30
565/565 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0012 - mae: 0.0220 - val_loss: 8.2579e-04 - val_mae: 0.0218
Epoch 7/30
565/565 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0011 - mae: 0.0215 - val_loss: 7.2072e-04 - val_mae: 0.0197
Epoch 8/30
565/565 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0011 - mae: 0.0213 - val_loss: 7.0234e-04 - val_mae: 0.0196
Epoch 9/30
565/565 ━━━━━

In [16]:
DATA_PATH = "Non_Null_Datasets/preprocessed_Raw_data_15Min_2024_site_5024_Alipur_Delhi_DPCC_15Min.csv"


In [8]:
# =====================================
# SINGLE-CELL COMPLETE CODE EXAMPLE
# =====================================

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# ------------------------------------------------
# 1. Configuration
# ------------------------------------------------

# Seven pollutants we want to predict (multi-output)
TARGET_POLLUTANTS = [
    "PM2.5 (µg/m³)", "PM10 (µg/m³)", "NO (µg/m³)",
    "NO2 (µg/m³)", "SO2 (µg/m³)", "CO (mg/m³)",
    "Ozone (µg/m³)"
]

# Meteorological columns we want to use as features
# (Excluding 'VWS (m/s)' since it's empty in your snippet)
MET_FEATURES = [
    "AT (°C)", "RH (%)", "WS (m/s)", "WD (deg)",
    "RF (mm)", "TOT-RF (mm)", "SR (W/mt2)",
    "BP (mmHg)"
]

# We can also include the target pollutants themselves as inputs,
# so the model sees past pollutant data to predict future pollutants.
# This is often beneficial to exploit pollutant inter-correlation.
ALL_INPUT_COLUMNS = TARGET_POLLUTANTS + MET_FEATURES

# Other hyperparams
N_STEPS = 4        # Past timesteps to use as input
TEST_SIZE_RATIO = 0.3
BATCH_SIZE = 8
EPOCHS = 20
SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

# ------------------------------------------------
# 2. Load & Basic Cleaning
# ------------------------------------------------
df = pd.read_csv(DATA_PATH, parse_dates=["Timestamp"], index_col="Timestamp")
print("Initial DF shape:", df.shape)

# Forward/backward fill missing data
df = df.ffill().bfill()

# Identify columns that are entirely NaN (if any remain) and drop them
cols_to_drop = [col for col in df.columns if df[col].isna().all()]
if cols_to_drop:
    print("Dropping columns that are entirely NaN:", cols_to_drop)
    df.drop(columns=cols_to_drop, inplace=True)

print("\nColumns after dropping all-NaN columns:")
print(df.columns.tolist())

# ------------------------------------------------
# 3. Build Input & Target DataFrames
# ------------------------------------------------
# Filter columns for input
input_cols = [c for c in ALL_INPUT_COLUMNS if c in df.columns]
df_input = df[input_cols].copy()

# Filter columns for target pollutants
target_cols = [c for c in TARGET_POLLUTANTS if c in df.columns]
df_target = df[target_cols].copy()

if len(target_cols) == 0:
    raise ValueError("None of the required target pollutant columns exist in the CSV!")

# Drop any rows that have NaNs in these columns
df_input.dropna(axis=0, how='any', inplace=True)
df_target.dropna(axis=0, how='any', inplace=True)

# Align indexes
common_index = df_input.index.intersection(df_target.index)
df_input = df_input.loc[common_index]
df_target = df_target.loc[common_index]

print("\ndf_input shape after cleaning:", df_input.shape)
print("df_target shape after cleaning:", df_target.shape)

if len(df_input) == 0:
    raise ValueError("No rows remain after cleaning. Check columns for too many NaNs.")

# ------------------------------------------------
# 4. Scaling
# ------------------------------------------------
# We'll scale inputs and targets separately
input_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

scaled_inputs = input_scaler.fit_transform(df_input.values)   # (samples, input_features)
scaled_targets = target_scaler.fit_transform(df_target.values) # (samples, #pollutants)

# ------------------------------------------------
# 5. Create Time-Series Sequences (Multi-Output)
# ------------------------------------------------
def create_multioutput_dataset(inputs, targets, n_steps):
    """
    inputs: 2D array (samples, input_features)
    targets: 2D array (samples, output_dim)
    Returns:
      X -> (num_samples - n_steps, n_steps, input_features)
      y -> (num_samples - n_steps, output_dim)
    """
    X_list, y_list = [], []
    for i in range(len(inputs) - n_steps):
        X_list.append(inputs[i : i + n_steps])
        y_list.append(targets[i + n_steps])
    return np.array(X_list), np.array(y_list)

X_all, y_all = create_multioutput_dataset(scaled_inputs, scaled_targets, N_STEPS)
print("\nSequence shapes:")
print("X_all:", X_all.shape, " y_all:", y_all.shape)

# ------------------------------------------------
# 6. Train-Test Split
# ------------------------------------------------
test_size = int(len(X_all) * TEST_SIZE_RATIO)
X_train, X_test = X_all[:-test_size], X_all[-test_size:]
y_train, y_test = y_all[:-test_size], y_all[-test_size:]

print("Train set shape:", X_train.shape, y_train.shape)
print("Test  set shape:", X_test.shape,  y_test.shape)

# ------------------------------------------------
# 7. Build Model (MLP + LSTM)
# ------------------------------------------------
model = tf.keras.Sequential([
    # Flatten the past timesteps for an MLP
    tf.keras.layers.Flatten(input_shape=(N_STEPS, len(input_cols))),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    # Reshape back into (batch, timesteps, features) for LSTM
    tf.keras.layers.Reshape((1, 64)),
    tf.keras.layers.LSTM(32, return_sequences=False),
    tf.keras.layers.Dropout(0.2),
    # Final: output dimension = number of target pollutants
    tf.keras.layers.Dense(len(target_cols))
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

# ------------------------------------------------
# 8. Train the Model
# ------------------------------------------------
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

# ------------------------------------------------
# 9. Evaluate on Test
# ------------------------------------------------
mse, mae = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest MSE: {mse:.4f},   Test MAE: {mae:.4f}")

# ------------------------------------------------
# 10. Predict & Inverse-Transform
# ------------------------------------------------
y_pred_scaled = model.predict(X_test)

y_test_unscaled = target_scaler.inverse_transform(y_test)
y_pred_unscaled = target_scaler.inverse_transform(y_pred_scaled)

# Example: RMSE per pollutant
rmse_each = np.sqrt(np.mean((y_test_unscaled - y_pred_unscaled) ** 2, axis=0))

print("\nRMSE per pollutant (original scale):")
for i, col in enumerate(target_cols):
    print(f"  {col}: {rmse_each[i]:.2f}")


Initial DF shape: (28224, 24)
Dropping columns that are entirely NaN: ['O Xylene (µg/m³)', 'WS (m/s)', 'VWS (m/s)']

Columns after dropping all-NaN columns:
['PM2.5 (µg/m³)', 'PM10 (µg/m³)', 'NO (µg/m³)', 'NO2 (µg/m³)', 'NOx (ppb)', 'NH3 (µg/m³)', 'SO2 (µg/m³)', 'CO (mg/m³)', 'Ozone (µg/m³)', 'Benzene (µg/m³)', 'Toluene (µg/m³)', 'Xylene (µg/m³)', 'Eth-Benzene (µg/m³)', 'MP-Xylene (µg/m³)', 'AT (°C)', 'RH (%)', 'WD (deg)', 'RF (mm)', 'TOT-RF (mm)', 'SR (W/mt2)', 'BP (mmHg)']

df_input shape after cleaning: (28224, 14)
df_target shape after cleaning: (28224, 7)

Sequence shapes:
X_all: (28220, 4, 14)  y_all: (28220, 7)
Train set shape: (19754, 4, 14) (19754, 7)
Test  set shape: (8466, 4, 14) (8466, 7)


c:\Users\DELL\Documents\Amrita\4th year\AirQuality_FederatedLearning\venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_1 (Flatten)             │ (None, 56)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         3,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_1 (Reshape)             │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │           231 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,295 (63.65 KB)

 Trainable params: 16,295 (63.65 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
1976/1976 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - loss: 0.0077 - mae: 0.0562 - val_loss: 0.0019 - val_mae: 0.0310
Epoch 2/20
1976/1976 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 0.0022 - mae: 0.0318 - val_loss: 0.0017 - val_mae: 0.0287
Epoch 3/20
1976/1976 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0019 - mae: 0.0295 - val_loss: 0.0013 - val_mae: 0.0257
Epoch 4/20
1976/1976 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.0018 - mae: 0.0289 - val_loss: 0.0012 - val_mae: 0.0239
Epoch 5/20
1976/1976 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - loss: 0.0018 - mae: 0.0282 - val_loss: 0.0011 - val_mae: 0.0232
Epoch 6/20
1976/1976 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - loss: 0.0017 - mae: 0.0278 - val_loss: 0.0011 - val_mae: 0.0225
Epoch 7/20
1976/1976 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - loss: 0.0017 - mae: 0.0276 - val_loss: 0.0011 - val_mae: 0.0222
Epoch 8/20
1976/1976 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0016 - mae: 0.0272 - val_loss: 0.0011 - val_mae: 0.0215
Epoch 9/20
1976/1976 ━━━━━━━━━━━━━━━━━━━

In [ ]:
import numpy as np

mse_each = np.mean((y_test_unscaled - y_pred_unscaled)**2, axis=0)

print("\nMSE per pollutant (original scale):")
for i, col in enumerate(target_cols):
    print(f"  {col}: {mse_each[i]:.2f}")

# CELL (e.g., at index 27)


# Compute per-pollutant MSE:
mse_each = np.mean((y_test_unscaled - y_pred_unscaled) ** 2, axis=0)

# Compute per-pollutant RMSE:
rmse_each = np.sqrt(mse_each)

print("\nMSE per pollutant (original scale):")
for i, col in enumerate(target_cols):
    print(f"  {col}: {mse_each[i]:.2f}")

print("\nRMSE per pollutant (original scale):")
for i, col in enumerate(target_cols):
    print(f"  {col}: {rmse_each[i]:.2f}")



MSE per pollutant (original scale):
  PM2.5 (µg/m³): 205.19
  PM10 (µg/m³): 803.30
  NO (µg/m³): 34.42
  NO2 (µg/m³): 12.42
  SO2 (µg/m³): 13.50
  CO (mg/m³): 0.08
  Ozone (µg/m³): 68.36

MSE per pollutant (original scale):
  PM2.5 (µg/m³): 205.19
  PM10 (µg/m³): 803.30
  NO (µg/m³): 34.42
  NO2 (µg/m³): 12.42
  SO2 (µg/m³): 13.50
  CO (mg/m³): 0.08
  Ozone (µg/m³): 68.36

RMSE per pollutant (original scale):
  PM2.5 (µg/m³): 14.32
  PM10 (µg/m³): 28.34
  NO (µg/m³): 5.87
  NO2 (µg/m³): 3.52
  SO2 (µg/m³): 3.67
  CO (mg/m³): 0.28
  Ozone (µg/m³): 8.27


### with time stamps as a feature

In [ ]:
# ===================================================================
# SINGLE-CELL CODE: Multi-Output LSTM with Pollutants + Time Features
# ===================================================================
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# -----------------------------
# 1. Configuration
# -----------------------------

POLLUTANTS = [
    "PM2.5 (µg/m³)", "PM10 (µg/m³)", "NO (µg/m³)",
    "NO2 (µg/m³)", "SO2 (µg/m³)", "CO (mg/m³)",
    "Ozone (µg/m³)"
]

# We'll add these columns for time-based features
TIME_COLS = ["hour", "dayofweek", "month"]

N_STEPS = 4         # Past timesteps to use as input
TEST_SIZE_RATIO = 0.2
BATCH_SIZE = 32
EPOCHS = 20
SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

# -----------------------------
# 2. Load Data
# -----------------------------
df = pd.read_csv(DATA_PATH, parse_dates=["Timestamp"], index_col="Timestamp")
df = df.ffill().bfill()  # fill small gaps

# Filter to just the pollutant columns we need
df = df[POLLUTANTS].copy()

# Drop any rows still containing NaN
df.dropna(inplace=True)

# -----------------------------
# 3. Create Timestamp Features
# -----------------------------
# We'll extract hour, day-of-week, month from the Timestamp index
df["hour"] = df.index.hour
df["dayofweek"] = df.index.dayofweek
df["month"] = df.index.month

# The final input columns = the original pollutants + the new time features
INPUT_COLUMNS = POLLUTANTS + TIME_COLS  # e.g., 7 pollutants + 3 time features = 10

# Because we’re forecasting the same pollutants at the next step,
# define targets = the original pollutant columns:
TARGET_COLUMNS = POLLUTANTS

# -----------------------------
# 4. Scale Inputs & Targets
# -----------------------------
# We'll separate input from target in the DataFrame
df_input = df[INPUT_COLUMNS].copy()
df_target = df[TARGET_COLUMNS].copy()

input_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

scaled_inputs = input_scaler.fit_transform(df_input.values)   # shape: (samples, #features)
scaled_targets = target_scaler.fit_transform(df_target.values) # shape: (samples, #pollutants)

# -----------------------------
# 5. Create Multi-Output Time Steps
# -----------------------------
def create_multioutput_dataset(X, y, n_steps):
    """
    X: 2D array of shape (samples, #features)
    y: 2D array of shape (samples, #pollutants)
    returns: X_array of shape (samples - n_steps, n_steps, #features)
             y_array of shape (samples - n_steps, #pollutants)
    """
    X_list, y_list = [], []
    for i in range(len(X) - n_steps):
        X_list.append(X[i : i + n_steps])
        y_list.append(y[i + n_steps])
    return np.array(X_list), np.array(y_list)

X_all, y_all = create_multioutput_dataset(scaled_inputs, scaled_targets, N_STEPS)

# -----------------------------
# 6. Train-Test Split
# -----------------------------
test_size = int(len(X_all) * TEST_SIZE_RATIO)
X_train, X_test = X_all[:-test_size], X_all[-test_size:]
y_train, y_test = y_all[:-test_size], y_all[-test_size:]

print("X_train shape:", X_train.shape, "| y_train shape:", y_train.shape)
print("X_test  shape:", X_test.shape,  "| y_test  shape:", y_test.shape)

# -----------------------------
# 7. Build Model
# -----------------------------
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(N_STEPS, len(INPUT_COLUMNS))),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Reshape((1, 64)),
    tf.keras.layers.LSTM(32, return_sequences=False),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(len(POLLUTANTS))
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

# -----------------------------
# 8. Train
# -----------------------------
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

# -----------------------------
# 9. Evaluate & Predict
# -----------------------------
mse, mae = model.evaluate(X_test, y_test, verbose=0)
print(f"Test MSE: {mse:.4f}, Test MAE: {mae:.4f}")

y_pred_scaled = model.predict(X_test)

# Inverse transform
y_test_unscaled = target_scaler.inverse_transform(y_test)
y_pred_unscaled = target_scaler.inverse_transform(y_pred_scaled)

# Example: per-pollutant RMSE
rmse_per_pollutant = np.sqrt(np.mean((y_test_unscaled - y_pred_unscaled)**2, axis=0))
for i, pollutant in enumerate(POLLUTANTS):
    print(f"{pollutant} RMSE: {rmse_per_pollutant[i]:.2f}")



X_train shape: (22576, 4, 10) | y_train shape: (22576, 7)
X_test  shape: (5644, 4, 10) | y_test  shape: (5644, 7)


c:\Users\DELL\Documents\Amrita\4th year\AirQuality_FederatedLearning\venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_2 (Flatten)             │ (None, 40)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         2,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_2 (Reshape)             │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 7)              │           231 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,271 (59.65 KB)

 Trainable params: 15,271 (59.65 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
565/565 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0095 - mae: 0.0653 - val_loss: 0.0017 - val_mae: 0.0312
Epoch 2/20
565/565 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0029 - mae: 0.0365 - val_loss: 0.0015 - val_mae: 0.0274
Epoch 3/20
565/565 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0022 - mae: 0.0313 - val_loss: 0.0013 - val_mae: 0.0261
Epoch 4/20
565/565 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0019 - mae: 0.0292 - val_loss: 0.0013 - val_mae: 0.0254
Epoch 5/20
565/565 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0018 - mae: 0.0281 - val_loss: 0.0012 - val_mae: 0.0240
Epoch 6/20
565/565 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0017 - mae: 0.0273 - val_loss: 0.0015 - val_mae: 0.0295
Epoch 7/20
565/565 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0017 - mae: 0.0269 - val_loss: 0.0013 - val_mae: 0.0283
Epoch 8/20
565/565 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0016 - mae: 0.0266 - val_loss: 0.0013 - val_mae: 0.0257
Epoch 9/20
565/565 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - lo

In [18]:
# ===================================================================
# SINGLE-CELL CODE: Multi-Output LSTM with Pollutants + Time Features
# ===================================================================
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# -----------------------------
# 1. Configuration
# -----------------------------

POLLUTANTS = [
    "PM2.5 (µg/m³)", "PM10 (µg/m³)", "NO (µg/m³)",
    "NO2 (µg/m³)", "SO2 (µg/m³)", "CO (mg/m³)",
    "Ozone (µg/m³)"
]

# We'll add these columns for time-based features
TIME_COLS = ["hour", "dayofweek", "month"]

N_STEPS = 4         # Past timesteps to use as input
TEST_SIZE_RATIO = 0.2
BATCH_SIZE = 32
EPOCHS = 20
SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

# -----------------------------
# 2. Load Data
# -----------------------------
df = pd.read_csv(DATA_PATH, parse_dates=["Timestamp"], index_col="Timestamp")
df = df.ffill().bfill()  # fill small gaps

# Filter to just the pollutant columns we need
df = df[POLLUTANTS].copy()

# Drop any rows still containing NaN
df.dropna(inplace=True)

# -----------------------------
# 3. Create Timestamp Features
# -----------------------------
# We'll extract hour, day-of-week, month from the Timestamp index
df["hour"] = df.index.hour
df["dayofweek"] = df.index.dayofweek
df["month"] = df.index.month

# The final input columns = the original pollutants + the new time features
INPUT_COLUMNS = POLLUTANTS + TIME_COLS  # e.g., 7 pollutants + 3 time features = 10

# Because we’re forecasting the same pollutants at the next step,
# define targets = the original pollutant columns:
TARGET_COLUMNS = POLLUTANTS

# -----------------------------
# 4. Scale Inputs & Targets
# -----------------------------
# We'll separate input from target in the DataFrame
df_input = df[INPUT_COLUMNS].copy()
df_target = df[TARGET_COLUMNS].copy()

input_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

scaled_inputs = input_scaler.fit_transform(df_input.values)   # shape: (samples, #features)
scaled_targets = target_scaler.fit_transform(df_target.values) # shape: (samples, #pollutants)

# -----------------------------
# 5. Create Multi-Output Time Steps
# -----------------------------
def create_multioutput_dataset(X, y, n_steps):
    """
    X: 2D array of shape (samples, #features)
    y: 2D array of shape (samples, #pollutants)
    returns: X_array of shape (samples - n_steps, n_steps, #features)
             y_array of shape (samples - n_steps, #pollutants)
    """
    X_list, y_list = [], []
    for i in range(len(X) - n_steps):
        X_list.append(X[i : i + n_steps])
        y_list.append(y[i + n_steps])
    return np.array(X_list), np.array(y_list)

X_all, y_all = create_multioutput_dataset(scaled_inputs, scaled_targets, N_STEPS)

# -----------------------------
# 6. Train-Test Split
# -----------------------------
test_size = int(len(X_all) * TEST_SIZE_RATIO)
X_train, X_test = X_all[:-test_size], X_all[-test_size:]
y_train, y_test = y_all[:-test_size], y_all[-test_size:]

print("X_train shape:", X_train.shape, "| y_train shape:", y_train.shape)
print("X_test  shape:", X_test.shape,  "| y_test  shape:", y_test.shape)

# -----------------------------
# 7. Build Model
# -----------------------------
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(N_STEPS, len(INPUT_COLUMNS))),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Reshape((1, 64)),
    tf.keras.layers.LSTM(32, return_sequences=False),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(len(POLLUTANTS))
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

# -----------------------------
# 8. Train
# -----------------------------
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

# -----------------------------
# 9. Evaluate & Predict
# -----------------------------
mse, mae = model.evaluate(X_test, y_test, verbose=0)
print(f"Test MSE: {mse:.4f}, Test MAE: {mae:.4f}")

y_pred_scaled = model.predict(X_test)

# Inverse transform
y_test_unscaled = target_scaler.inverse_transform(y_test)
y_pred_unscaled = target_scaler.inverse_transform(y_pred_scaled)

# Example: per-pollutant RMSE
rmse_per_pollutant = np.sqrt(np.mean((y_test_unscaled - y_pred_unscaled)**2, axis=0))
for i, pollutant in enumerate(POLLUTANTS):
    print(f"{pollutant} RMSE: {rmse_per_pollutant[i]:.2f}")



X_train shape: (22576, 4, 10) | y_train shape: (22576, 7)
X_test  shape: (5644, 4, 10) | y_test  shape: (5644, 7)


c:\Users\DELL\Documents\Amrita\4th year\AirQuality_FederatedLearning\venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_3 (Flatten)             │ (None, 40)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         2,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_3 (Reshape)             │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 7)              │           231 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,271 (59.65 KB)

 Trainable params: 15,271 (59.65 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
565/565 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0101 - mae: 0.0654 - val_loss: 0.0014 - val_mae: 0.0273
Epoch 2/20
565/565 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0028 - mae: 0.0356 - val_loss: 0.0013 - val_mae: 0.0267
Epoch 3/20
565/565 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0021 - mae: 0.0309 - val_loss: 0.0013 - val_mae: 0.0258
Epoch 4/20
565/565 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0019 - mae: 0.0291 - val_loss: 0.0013 - val_mae: 0.0251
Epoch 5/20
565/565 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0018 - mae: 0.0282 - val_loss: 0.0012 - val_mae: 0.0243
Epoch 6/20
565/565 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0017 - mae: 0.0277 - val_loss: 0.0010 - val_mae: 0.0219
Epoch 7/20
565/565 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0017 - mae: 0.0271 - val_loss: 0.0011 - val_mae: 0.0236
Epoch 8/20
565/565 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0016 - mae: 0.0269 - val_loss: 0.0011 - val_mae: 0.0236
Epoch 9/20
565/565 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - lo

### Multi-Pollutant Correlation Model


In [20]:
# %% Cell 1: Multi-Pollutant Correlation Model
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# Configuration
SEED = 42
DATA_PATH = "Non_Null_Datasets/preprocessed_Raw_data_15Min_2024_site_5024_Alipur_Delhi_DPCC_15Min.csv"
POLLUTANTS = ['PM2.5 (µg/m³)', 'PM10 (µg/m³)', 'NO (µg/m³)', 'NO2 (µg/m³)', 
             'SO2 (µg/m³)', 'CO (mg/m³)', 'Ozone (µg/m³)']
N_STEPS = 24  # 6-hour window (24*15min)
BATCH_SIZE = 64

# Data Loading & Preprocessing
df = pd.read_csv(DATA_PATH, parse_dates=['Timestamp'], index_col='Timestamp')
df = df[POLLUTANTS].resample('15T').mean().ffill()

# Create 3D sequences [samples, timesteps, features]
def create_dataset(data, n_steps):
    X, y = [], []
    for i in range(len(data)-n_steps):
        X.append(data[i:i+n_steps])
        y.append(data[i+n_steps])
    return np.array(X), np.array(y)

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df)
X, y = create_dataset(scaled_data, N_STEPS)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Correlation-Aware Hybrid Model
model = tf.keras.Sequential([
    tf.keras.layers.Conv1D(64, 3, activation='relu', input_shape=(N_STEPS, len(POLLUTANTS))),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(128, return_sequences=True)),
    tf.keras.layers.LSTM(64, return_sequences=True),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(len(POLLUTANTS))
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
history = model.fit(X_train, y_train, epochs=100, batch_size=BATCH_SIZE, 
                   validation_split=0.2, verbose=1)

# Evaluate
test_loss, test_mae = model.evaluate(X_test, y_test)
print(f"Test MSE: {test_loss:.4f}, MAE: {test_mae:.4f}")

C:\Users\DELL\AppData\Local\Temp\ipykernel_20536\3260512141.py:18: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df = df[POLLUTANTS].resample('15T').mean().ffill()
c:\Users\DELL\Documents\Amrita\4th year\AirQuality_FederatedLearning\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
282/282 ━━━━━━━━━━━━━━━━━━━━ 12s 29ms/step - loss: 0.0068 - mae: 0.0496 - val_loss: 6.3064e-04 - val_mae: 0.0184
Epoch 2/100
282/282 ━━━━━━━━━━━━━━━━━━━━ 8s 28ms/step - loss: 0.0012 - mae: 0.0221 - val_loss: 5.7599e-04 - val_mae: 0.0182
Epoch 3/100
282/282 ━━━━━━━━━━━━━━━━━━━━ 8s 27ms/step - loss: 9.6670e-04 - mae: 0.0193 - val_loss: 5.4878e-04 - val_mae: 0.0177
Epoch 4/100
282/282 ━━━━━━━━━━━━━━━━━━━━ 8s 28ms/step - loss: 8.6184e-04 - mae: 0.0181 - val_loss: 6.7383e-04 - val_mae: 0.0204
Epoch 5/100
282/282 ━━━━━━━━━━━━━━━━━━━━ 8s 28ms/step - loss: 7.8359e-04 - mae: 0.0171 - val_loss: 6.9605e-04 - val_mae: 0.0208
Epoch 6/100
282/282 ━━━━━━━━━━━━━━━━━━━━ 8s 27ms/step - loss: 7.3738e-04 - mae: 0.0165 - val_loss: 8.1752e-04 - val_mae: 0.0226
Epoch 7/100
282/282 ━━━━━━━━━━━━━━━━━━━━ 9s 32ms/step - loss: 6.9478e-04 - mae: 0.0159 - val_loss: 7.4423e-04 - val_mae: 0.0210
Epoch 8/100
282/282 ━━━━━━━━━━━━━━━━━━━━ 9s 33ms/step - loss: 6.6190e-04 - mae: 0.0153 - val_loss: 5.7578e-04 -

### multi pollution with weather 

In [23]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Get predictions from the trained model
predictions = model.predict(X_test)

# Inverse transform to get back to original pollutant scales
predictions_inversed = scaler.inverse_transform(predictions)
y_test_inversed = scaler.inverse_transform(y_test)

# Calculate MSE, MAE, and RMSE for each pollutant
results_list = []
for i, pollutant in enumerate(POLLUTANTS):
    y_true = y_test_inversed[:, i]
    y_pred = predictions_inversed[:, i]
    
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mse)

    results_list.append([pollutant, mse, mae, rmse])

# Display results in a DataFrame
metrics_df = pd.DataFrame(results_list, columns=['Pollutant', 'MSE', 'MAE', 'RMSE'])
print(metrics_df)

# Example: Forecast plot for the first pollutant
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(range(len(y_true)), y_true, label='Actual')
plt.plot(range(len(y_pred)), y_pred, label='Predicted')
plt.title(f'Forecast vs Actual: {POLLUTANTS[0]}')
plt.xlabel('Time steps')
plt.ylabel('Concentration')
plt.legend()
plt.tight_layout()
plt.show()


177/177 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step


ValueError: operands could not be broadcast together with shapes (5640,7) (16,) (5640,7) 

In [22]:
# %% Cell 2: Weather-Integrated Pollution Model
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# Configuration
SEED = 42
DATA_PATH = "Non_Null_Datasets/preprocessed_Raw_data_15Min_2024_site_5024_Alipur_Delhi_DPCC_15Min.csv"
POLLUTANTS = ['PM2.5 (µg/m³)', 'PM10 (µg/m³)', 'NO (µg/m³)', 'NO2 (µg/m³)', 
             'SO2 (µg/m³)', 'CO (mg/m³)', 'Ozone (µg/m³)']
WEATHER_FEATURES = ['AT (°C)', 'RH (%)', 'WS (m/s)', 'WD (deg)', 
                   'RF (mm)', 'TOT-RF (mm)', 'SR (W/mt2)', 'BP (mmHg)', 'VWS (m/s)']
N_STEPS = 24  # 6-hour window
BATCH_SIZE = 64

# Data Loading & Preprocessing
df = pd.read_csv(DATA_PATH, parse_dates=['Timestamp'], index_col='Timestamp')
df = df[POLLUTANTS + WEATHER_FEATURES].resample('15T').mean().ffill()

# Create 3D sequences
def create_dataset(data, n_steps):
    X, y = [], []
    for i in range(len(data)-n_steps):
        X.append(data[i:i+n_steps])
        y.append(data[i+n_steps, :len(POLLUTANTS)])  # Predict only pollutants
    return np.array(X), np.array(y)

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df)
X, y = create_dataset(scaled_data, N_STEPS)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Weather-Aware Hybrid Model
input_layer = tf.keras.layers.Input(shape=(N_STEPS, len(POLLUTANTS)+len(WEATHER_FEATURES)))
x = tf.keras.layers.Conv1D(64, 3, activation='relu')(input_layer)
weather_branch = tf.keras.layers.Dense(32, activation='relu')(x)
pollutant_branch = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(128, return_sequences=True))(x)
merged = tf.keras.layers.Concatenate()([weather_branch, pollutant_branch])
x = tf.keras.layers.Attention()([merged, merged])
x = tf.keras.layers.Flatten()(x)
output = tf.keras.layers.Dense(len(POLLUTANTS))(x)

model = tf.keras.Model(inputs=input_layer, outputs=output)
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
history = model.fit(X_train, y_train, epochs=100, batch_size=BATCH_SIZE,
                   validation_split=0.2, verbose=1)

# Evaluate
test_loss, test_mae = model.evaluate(X_test, y_test)
print(f"Test MSE: {test_loss:.4f}, MAE: {test_mae:.4f}")

C:\Users\DELL\AppData\Local\Temp\ipykernel_20536\3550112184.py:20: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df = df[POLLUTANTS + WEATHER_FEATURES].resample('15T').mean().ffill()
c:\Users\DELL\Documents\Amrita\4th year\AirQuality_FederatedLearning\venv\Lib\site-packages\sklearn\utils\_array_api.py:769: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmin(X, axis=axis))
c:\Users\DELL\Documents\Amrita\4th year\AirQuality_FederatedLearning\venv\Lib\site-packages\sklearn\utils\_array_api.py:786: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmax(X, axis=axis))


Epoch 1/100
282/282 ━━━━━━━━━━━━━━━━━━━━ 11s 26ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 2/100
282/282 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 3/100
282/282 ━━━━━━━━━━━━━━━━━━━━ 7s 26ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 4/100
282/282 ━━━━━━━━━━━━━━━━━━━━ 7s 26ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 5/100
282/282 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 6/100
282/282 ━━━━━━━━━━━━━━━━━━━━ 7s 26ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 7/100
282/282 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 8/100
282/282 ━━━━━━━━━━━━━━━━━━━━ 7s 24ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 9/100
282/282 ━━━━━━━━━━━━━━━━━━━━ 7s 26ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 10/100
282/282 ━━━━━━

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Get predictions from the trained model
predictions = model.predict(X_test)

# Prepare arrays for inverse transform
num_pollutants = len(POLLUTANTS)
num_weather = len(WEATHER_FEATURES)

predictions_full = np.zeros((len(predictions), num_pollutants + num_weather))
predictions_full[:, :num_pollutants] = predictions

y_test_full = np.zeros((len(y_test), num_pollutants + num_weather))
y_test_full[:, :num_pollutants] = y_test

# Inverse transform only the pollutant portion
predictions_inversed = scaler.inverse_transform(predictions_full)[:, :num_pollutants]
y_test_inversed = scaler.inverse_transform(y_test_full)[:, :num_pollutants]

# Calculate MSE, MAE, and RMSE for each pollutant
results_list = []
for i, pollutant in enumerate(POLLUTANTS):
    y_true = y_test_inversed[:, i]
    y_pred = predictions_inversed[:, i]
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    results_list.append([pollutant, mse, mae, rmse])

# Display results
metrics_df = pd.DataFrame(results_list, columns=['Pollutant', 'MSE', 'MAE', 'RMSE'])
print(metrics_df)

# Example forecast plot for the first pollutant
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(range(len(y_true)), y_true, label='Actual')
plt.plot(range(len(y_pred)), y_pred, label='Predicted')
plt.title(f'Forecast vs Actual: {POLLUTANTS[0]}')
plt.xlabel('Time Steps')
plt.ylabel('Concentration')
plt.legend()
plt.tight_layout()
plt.show()


### now with time features as cyclic 

In [ ]:
# %% Cell 1: Multi-Pollutant Model with Time Features
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# Configuration
SEED = 42
DATA_PATH = "Non_Null_Datasets/preprocessed_Raw_data_15Min_2024_site_5024_Alipur_Delhi_DPCC_15Min.csv"
POLLUTANTS = ['PM2.5 (µg/m³)', 'PM10 (µg/m³)', 'NO (µg/m³)', 'NO2 (µg/m³)', 
             'SO2 (µg/m³)', 'CO (mg/m³)', 'Ozone (µg/m³)']
N_STEPS = 24  # 6-hour window (24*15min)
BATCH_SIZE = 64

# Load and preprocess data with timestamp features
def load_data_with_time_features(file_path):
    df = pd.read_csv(file_path, parse_dates=['Timestamp'], index_col='Timestamp')
    df = df[POLLUTANTS].resample('15T').mean().ffill()
    
    # Extract time-based features
    df['hour'] = df.index.hour
    df['day_of_week'] = df.index.dayofweek
    df['month'] = df.index.month
    
    # Cyclical encoding for time features
    df['hour_sin'] = np.sin(2 * np.pi * df['hour']/24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour']/24)
    df['day_sin'] = np.sin(2 * np.pi * df['day_of_week']/7)
    df['day_cos'] = np.cos(2 * np.pi * df['day_of_week']/7)
    df['month_sin'] = np.sin(2 * np.pi * (df['month']-1)/12)
    df['month_cos'] = np.cos(2 * np.pi * (df['month']-1)/12)
    
    return df.drop(['hour', 'day_of_week', 'month'], axis=1)

# Create 3D sequences [samples, timesteps, features]
def create_dataset(data, n_steps):
    X, y = [], []
    for i in range(len(data)-n_steps):
        X.append(data.iloc[i:i+n_steps].values)
        y.append(data.iloc[i+n_steps][POLLUTANTS].values)
    return np.array(X), np.array(y)

# Data processing
df = load_data_with_time_features(DATA_PATH)
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df)
scaled_df = pd.DataFrame(scaled_data, columns=df.columns, index=df.index)

X, y = create_dataset(scaled_df, N_STEPS)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False, random_state=SEED)

# Temporal Correlation Model
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(N_STEPS, scaled_df.shape[1])),
    tf.keras.layers.Conv1D(64, 3, activation='relu'),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(128, return_sequences=True)),
    tf.keras.layers.Attention()([tf.keras.layers.LSTM(64, return_sequences=True)]*2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(len(POLLUTANTS))
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
history = model.fit(X_train, y_train, epochs=100, batch_size=BATCH_SIZE,
                   validation_split=0.2, verbose=1,
                   callbacks=[tf.keras.callbacks.EarlyStopping(patience=10)])

# Evaluate
test_loss, test_mae = model.evaluate(X_test, y_test)
print(f"Test MSE: {test_loss:.4f}, MAE: {test_mae:.4f}")

In [ ]:
# %% Cell 2: Weather+Time Integrated Model
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# Configuration
SEED = 42
DATA_PATH = "Non_Null_Datasets/preprocessed_Raw_data_15Min_2024_site_5024_Alipur_Delhi_DPCC_15Min.csv"
POLLUTANTS = ['PM2.5 (µg/m³)', 'PM10 (µg/m³)', 'NO (µg/m³)', 'NO2 (µg/m³)', 
             'SO2 (µg/m³)', 'CO (mg/m³)', 'Ozone (µg/m³)']
WEATHER_FEATURES = ['AT (°C)', 'RH (%)', 'WS (m/s)', 'WD (deg)', 
                   'RF (mm)', 'TOT-RF (mm)', 'SR (W/mt2)', 'BP (mmHg)', 'VWS (m/s)']
N_STEPS = 24
BATCH_SIZE = 64

def load_full_data(file_path):
    df = pd.read_csv(file_path, parse_dates=['Timestamp'], index_col='Timestamp')
    df = df[POLLUTANTS + WEATHER_FEATURES].resample('15T').mean().ffill()
    
    # Time features
    df['hour_sin'] = np.sin(2 * np.pi * df.index.hour/24)
    df['hour_cos'] = np.cos(2 * np.pi * df.index.hour/24)
    df['day_sin'] = np.sin(2 * np.pi * df.index.dayofweek/7)
    df['day_cos'] = np.cos(2 * np.pi * df.index.dayofweek/7)
    df['month_sin'] = np.sin(2 * np.pi * (df.index.month-1)/12)
    df['month_cos'] = np.cos(2 * np.pi * (df.index.month-1)/12)
    
    return df

def create_dataset(data, n_steps):
    X, y = [], []
    for i in range(len(data)-n_steps):
        X.append(data.iloc[i:i+n_steps].values)
        y.append(data.iloc[i+n_steps][POLLUTANTS].values)
    return np.array(X), np.array(y)

# Data processing
df = load_full_data(DATA_PATH)
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df)
scaled_df = pd.DataFrame(scaled_data, columns=df.columns, index=df.index)

X, y = create_dataset(scaled_df, N_STEPS)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False, random_state=SEED)

# Hybrid Weather-Time Model
input_layer = tf.keras.layers.Input(shape=(N_STEPS, scaled_df.shape[1]))
x = tf.keras.layers.Conv1D(64, 3, activation='relu')(input_layer)

# Time feature processing
time_features = tf.keras.layers.Dense(32, activation='relu')(x)

# Weather-pollutant processing
weather_pollutant = tf.keras.layers.Bidirectional(
    tf.keras.layers.LSTM(128, return_sequences=True))(x)

# Merge branches
merged = tf.keras.layers.Concatenate()([time_features, weather_pollutant])
x = tf.keras.layers.Attention()([merged, merged])
x = tf.keras.layers.Flatten()(x)
output = tf.keras.layers.Dense(len(POLLUTANTS))(x)

model = tf.keras.Model(inputs=input_layer, outputs=output)
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
history = model.fit(X_train, y_train, epochs=100, batch_size=BATCH_SIZE,
                   validation_split=0.2, verbose=1,
                   callbacks=[tf.keras.callbacks.EarlyStopping(patience=10)])

# Evaluate
test_loss, test_mae = model.evaluate(X_test, y_test)
print(f"Test MSE: {test_loss:.4f}, MAE: {test_mae:.4f}")